# Generador local del Tablero de Carnes
Lee el Excel y el JSON generados por `Tablero_Carnes_local.ipynb` y escribe el HTML en la carpeta de salida. Ejecutá todas las celdas en orden.
Las rutas y parámetros se comparten con el primer notebook mediante el Excel de configuración.


In [ ]:
from entorno_local import archivo_config, ruta_local


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# ============================================================================
#  NOTEBOOK 2 · Generador del Tablero de Fiscalización de Carnes (HTML unificado)
#  --------------------------------------------------------------------------
#  Entradas (outputs de Notebook 1):
#     - datos_tablero.json                      -> vistas Macro y por Planta
#     - tablero_fiscalizacion_indicadores.xlsx  -> pestaña Control (tabla + drawer)
#  Salida:
#     - Tablero_Carnes_Unificado.html  (3 pestañas: Control / Macro / Planta)
#
#  Colores/score de Macro y Planta se calculan EN EL NAVEGADOR (JS) desde los
#  valores crudos + umbrales del JSON. La pestaña Control usa los colores ya
#  calculados por Notebook 1 (Excel). No usa ningún HTML como entrada.
# ============================================================================
import json, math, os, re
import pandas as pd

# ------------------------- CONFIG: parámetros y rutas -------------------------
# XLSX, JSON_, PADRON_TOTAL y demás ya no están sueltos acá: se leen del mismo
# config_parametros_tablero.xlsx que usa Notebook 1 (hojas Rutas y Parametros_Generales),
# así los dos notebooks quedan mirando siempre la misma fuente de parámetros.
# Poné acá la carpeta INPUT (o directamente la ruta completa al archivo, las dos formas andan)
CONFIG_XLSX = archivo_config()
_df_rutas = pd.read_excel(CONFIG_XLSX, sheet_name='Rutas')
_df_gen   = pd.read_excel(CONFIG_XLSX, sheet_name='Parametros_Generales')
RUTAS  = dict(zip(_df_rutas['Parametro'].astype(str), _df_rutas['Valor']))
PARAMS = dict(zip(_df_gen['Parametro'].astype(str), _df_gen['Valor']))

OUTPUT_DIR   = ruta_local(RUTAS['OUTPUT_DIR'], 'OUTPUT_DIR')
XLSX  = os.path.join(OUTPUT_DIR, RUTAS['NOMBRE_EXCEL_SALIDA'])
JSON_ = os.path.join(OUTPUT_DIR, RUTAS['NOMBRE_JSON_SALIDA'])
OUT   = os.path.join(OUTPUT_DIR, RUTAS['NOMBRE_HTML_SALIDA'])
PADRON_TOTAL = int(PARAMS['PADRON_TOTAL'])   # total del padrón SIOCAL (las que no faenaron no están en el JSON)
# Dependencias instaladas mediante requirements.txt.
# -----------------------------------------------------------------

for entrada in (XLSX, JSON_):
    if not os.path.isfile(entrada):
        raise FileNotFoundError(f"Falta la entrada {entrada}. Ejecutá primero ejecutar_tablero.py --etapa datos.")
os.makedirs(os.path.dirname(OUT), exist_ok=True)


## Plantillas embebidas (el molde: CSS/markup/JS de las 3 pestañas)

In [ ]:
# ===================== PLANTILLAS EMBEBIDAS =====================
CSS_SHELL = r'''
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'Montserrat',sans-serif;background:#f0f2f5;color:#1a1f36;font-size:13px}
.apphdr{background:#242c50;border-bottom:3px solid #eab000;height:64px;display:flex;align-items:center;gap:14px;padding:0 22px;position:sticky;top:0;z-index:60}
.apphdr img{height:44px}
.apphdr .t{color:#fff;font-family:'Lora',serif;font-size:16px;font-weight:600;line-height:1.1}
.apphdr .s{color:#fff;font-size:9px;font-weight:300;text-transform:uppercase;letter-spacing:.06em;opacity:.85}
.topnav{display:flex;background:#242c4f;position:sticky;top:64px;z-index:55;padding-left:8px}
.topnav .tab{color:#cbd5e1;padding:11px 26px;font-size:13px;font-weight:500;cursor:pointer;border:none;background:none;font-family:'Montserrat';border-bottom:3px solid transparent}
.topnav .tab.on{color:#fff;border-bottom-color:#eab000;background:rgba(255,255,255,.05)}
.pane{display:none}.pane.on{display:block}
#ctrl-toolbar{display:flex;align-items:center;gap:12px;padding:10px 22px 0}
.lnk-planta{color:#242c4f;font-weight:500;cursor:pointer;text-decoration:none;border-bottom:1px dotted #94a3b8}
.lnk-planta:hover{color:#dc2626;border-bottom-color:#dc2626}
.mm{padding:18px 22px 60px}
'''
LOGO = r"""data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD/4gHYSUNDX1BST0ZJTEUAAQEAAAHIAAAAAAQwAABtbnRyUkdCIFhZWiAH4AABAAEAAAAAAABhY3NwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAQAA9tYAAQAAAADTLQAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAlkZXNjAAAA8AAAACRyWFlaAAABFAAAABRnWFlaAAABKAAAABRiWFlaAAABPAAAABR3dHB0AAABUAAAABRyVFJDAAABZAAAAChnVFJDAAABZAAAAChiVFJDAAABZAAAAChjcHJ0AAABjAAAADxtbHVjAAAAAAAAAAEAAAAMZW5VUwAAAAgAAAAcAHMAUgBHAEJYWVogAAAAAAAAb6IAADj1AAADkFhZWiAAAAAAAABimQAAt4UAABjaWFlaIAAAAAAAACSgAAAPhAAAts9YWVogAAAAAAAA9tYAAQAAAADTLXBhcmEAAAAAAAQAAAACZmYAAPKnAAANWQAAE9AAAApbAAAAAAAAAABtbHVjAAAAAAAAAAEAAAAMZW5VUwAAACAAAAAcAEcAbwBvAGcAbABlACAASQBuAGMALgAgADIAMAAxADb/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCACkAcMDASIAAhEBAxEB/8QAHAABAAIDAQEBAAAAAAAAAAAAAAYHBAUIAwIB/8QATRAAAQMDAwIDBgIGBQgHCQAAAQIDBAAFEQYSIQcxEyJBCBQVUWFxIzIWQlWBkdJSU2KVoSQ3cnWTlLGyGDM0gsPR1BdFVFaSo7PB4f/EABkBAQEBAQEBAAAAAAAAAAAAAAABAgMEBf/EACcRAQEAAgICAgEEAgMAAAAAAAABAhEDEiExE0FhBCLB8BQyUYGx/9oADAMBAAIRAxEAPwDnylKV955ClKUCleiWH1RlSUsuFlKghTgSdoUeQCfnwf4VJExo9nvMW4xUOSLO+AhSnME7VpKVBWOAfzY+3zBxLdCOiLIMNUwNK93SsNlfpuIzivQ2+WDEHhAmYjxGAFg7k71IyefLyhXfHbPYg1v2LepDV4sqytSGE+8NLBBA5ABwD3JKB3O0FVZtvD6bxZHBFCFRohawoeVSVNBaSR/aU+B9dw+dTsukNfYeYfdYdbUlxpRS4n+iQcH/ABrzqSRmk2+ySvd1l2TcnPdo6hlKiyFcqAxyFFOCM8fvrEulrZFwj2q1pXLl7fxFoOUrJ5G0dwMcnOO+CBjJu0001K/VpKFqQruk4NflUKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKJxuGTgetK2Vo98jNPT27YmXFwWXVOsFbaScEc/qq7EEH/AkUo3VujTrQt5cIt3S0yhsdbSofjpHOdnfcO+R2/iK94UNAS6xDW49bpSCpLZ5UjsDgevpx/SSkd8Z/GWITL4kNMTrK85v/yZ9J8J0Z4bBUU5HAyoqByRgZFbBMNlqQ46l9e5OXXEoXsfb/D37w05t34HBVwFD9XkGuVrT4hwvHih51TokRo5UFspBS4ykgO43KwrCDvwBklw5xivdq3b4C1NSksyEvpirdUoBCUIg7lKB+hbBB/spxUfffu90UHIS0R20hWVmSllLilApWtG8pIChwRzznPfFa6JbrxNCowUUIQrtIkJZRuwBwVkDO3H1x9Kuvym0ml7WUxpKY+5xLQUyxuADQWnyk+oIQlO5XptByCoEYTrjzCnY9uKnLjMyZUpKCkISrk4AHAPyHp9SAnItdxlPzPc7m3GRKK/G3P7UtPEjhbi+3hoCEkJGQogd+K/X1OuhxTs8oiEq3PEZLh3BKjnkAnBOWgs+bBJ5NRUbvUa3Q8RYzrr8ptWHnMjZ9gPmDWsqVsAyGVRdO2HIeC2/e5QBUrgEgFR2hQGRkdwc4BqMSWXI0hyO8na42opWM5wR3rpKy86UpVClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUClKUCt1Yg2mE64vUbNvBWQYxbdWteBwoAJKPUjlQI5rS1s7GbcoqakwJEyUpafAQ2o4Uf6JAwefmD+6pfQkERyO0oR1XdmWw6n8XDTbX17+IM9gfMCM+hrx1JvkWJMl3xvCSvDA8VlDaSe+1CeTwOdo74zWxNsuXuyJr8OHbIHjBhITI/ESoHlBCVJAXgE+fbwPmRnfu2ywRIgRqy8RLQZDKXRAaZW5KfBSCA66UKLIyMjyk4PGRg1z7SNaRnp9r266cjqtUSNbHESSWw7KRw3vKckn5ApHJ5Az6GtpE1lM09P+NQU2p91CEtuNL8Mh04GVJSknA823HAwhHHGB+KvFthnZp66aPtDY4Sv4fIkPn6lx1lRB/0do+lfI1Jdcq8TqBYHkq/Mh22OLQfptMfGPpUs350Ic/cZV61GzJdab8Rx1KEtoQCACe2FHB7+p/fUuuTU9DTUmQmVFQpa9xfbaexyNxCfEShPPySn7mshhzQ91eSNRXGwxHBgibaI0hpYPzUyWvDWPoAj7183u2NQhHn2u522Zblq2RpzMcNIc2JBIWDhTS++RuCj6DBBNt34GiubLtxeaRJ1LBbSv8AOVlKUgehV4ZUVevfJFRi4R0RZjkdEpmUlGMOs52K4zxkA/TtUt1BCl218vSbNAloSkLedYdS4Ak42q8oygHIwVjnNRCY407LddYYDDS1koaCiQgegyeTit4pXlSlK0hSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKBSlKDItkN643KLb4+3xpLyGW9xwNyiAMn7mrRs9qtDFlYiqaZge8+8Nx5qiUynE7WSp1WAfDRsLnBAwlQPPJP10y07bXGbI9KYQJrc1iW0oupaW+oupSWQpXl2oQtl0/recpA5yn0uchq86gVcLiq1N2RERdxnm3NupK46XsIZHinguO4Hlxkr3EkAVxyy3dNSaa5l+02C1x7jNV7nPca3WOKW/FEZoqOZTnA3OLI8uQOADgAIFQuW3bJcl2VK1A+8+6orccXGUpSlHkkknk1lX+5We93iTdJsq5l59e4hMZsJQOwSkb+EgAAD0AApp+16cut1ahGfc2EKCluOGMg7EJSVKOAvngHitSa81GLGtdskvJZj3R95xRwEIhKJP7gazLtpYWlKFXR24Qgv8pftziAftmsqVcLhGtKVWVoaetDxKEOl7EiX3BKlDzrTwQQkBAOAeeTr9Lapm2V9ceRm5WiQdsy3vrJafSe5H9FY7pWOQf3g3z9Hhie5WX9tL/3RX/nW40xcrbZXnWzdjKt8oBEyE5FV4b6Przwodwocg9q/NV2XTdouSUMTro/BlNJkwnxHRhxlXb9f8wIKVD0UlQ9K1GzTn/xN1/3dv+enuHpYVut1rauUaa1LL9qaYW/b5ayrc9HB/FjOoSk7igKOQMeUk8jGNR1E0/FTG9+t0JphEWJHWt5hwKZkNqQ0N/H5VhbiQRwSFA4ByT86HnWt1Dumok+4NvTHEuW911CUBiYkfhq3BRwFfkPphQPoKlenC5d9KSdOznLJFiz2G1SQ0XESYu2WhB4WfDGHFFXhAgnJ2hO4Z523G7a9qWpUo6iQYbFzamW6M1GiSElKWmiSlO3BQQTydzSmV5Pqs9u1Reu0u5tgpSlUKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKz9PQ2Z95jRpPi+7FW6QWsbw0kblkZ4yEg1gVv8AQ6ZzVzduESMqQ1GZWJSEKAWWXElte0HudqlHgHGCewNS+iLMhQGI2n4EVUBu4TYDykxJ78n3dhlD6UrbKxhRW6EN4ISQWwR2UncmLXK5wI2hoyb1ClPTNQSPfliHJTHDcZjcww3gtr8u4PYHHCUd+K3HUdxVmN3jxZVzubzj6nXHnAfBhpk7ylG7JBXsdcBxwVOK/o5Mc6mTYcXVj1mVao74s7LNtCluOAlTDaULJAUACVhZP1JrjjNtVovedKfsa9/3q1/6etlpiRZFait6LVbb8xOXIQhhxF2aBSonAP8A2f61pff4H7Dif7Z7+esm1XyNbbnGuEeyQw9GdS6g+K93Scj9eutnhG01rd4mpNQOvybm6zb7ehMKEhTfiPKZRuwQAEpGTkkZSBuwBgYGtjXSwQk7WdNM3BWBly4SXeT6kJaUjA+hKsfM1JYV30GmOHLZBNtuLhJW5dY4ntIJP6uCAB9C2o/WrN6svWy0aI0JKj3Cwwvf7cp2SpuxNue+KBQQtKC2AAM9jt71zuWtTS635VlDvFg1JpN60TdNpiv2hLk2F8PmLQVtqUnxWsuh3gf9YPlhfz5i/vOlP2Ne/wC9Wv8A09SibrHS8SMsWnTTUm4rZcZXcHUe6hSHEFCx4DStv5VHnP8AwqH+/wAD9hxP9s9/PWsZ+ErKRL0qhaVotF7CknIIurXB/wB3qeuSviF8FyYjRX7VeISri5CmTQ2S7u8KSkObQlK1qRvPAAG0jaUpIrj3+B+w4n+2e/nqWW25od0NHlRra0j4VdiythClK8ZqYypKxlRO3hjGfmvI7UyhK3nUnT7MmIX0Jkw2oUJEWBBUMrQ+y0hcjxl7QF/hjyqAG4AdtihVS1ctwS65Y51xiybjIuUuT8OTCmI8P3NZZkIfU5uOEoKX1LyMAKWrOMc0/MjvRJb0WQjY8ystuJznCgcEU4r40uTyqV9LdJRNa6mTZJWoYljU4j8B2QgrDrhUlKW0gEeY5/wqKVJ+kv8AnR0t/reL/wDlTW899bpJ7Sbrh0euvS5NtflXJm5xZ5WgPNMlvw1pwdpBJ7g5HPoflUR6e6Wna01lbdM29Ybfmu7fEUMhtIBUpZHrhIJx61151MjI6odHtZWtP4t205dpXg7sZ3MrK0gfdpZQPrVGdAVo0dovVfU6Qkh5htFptZ2jJfdIK1Jz6pTtP2Kq8+HNleO79tXGbY3WLom302saJ1w1nClTHj/k0JMVSFvAEBRB3EDGc81i6h6e6Rg9B7XreHqfx77JdSl6EXUbeSQUBH5gpPBJyftzVle3XtNy0e68FFpTL+7HcjLef+NaTXugunkb2cU660tb7gmRKkNJadnP7nEDxChQwk7ecGphyW44233Vsm7pz3Sum+lHTvpRe+it11pOtl3dXBiPImLcfG5DjbQW4tlIOPXy7v3176E0R0m6r9Or4NLaal2S8WxG1txclTrm8pKm1HJwoKKSCMfPGOK3f1GM34vhnpVSez/orTGudVy7Zqi9rtcZmGp5ooeQ2pxYIGMrBGACSRj0qC36JHgXudBiTETY8eQtpqQgeV1KVEBY+hAzVy+yronQmuLrLt2ooNxk3OJukgJe2RlMjYkJIHmKtxV9MVkdJ+ltg1V1c1rDmRXHbXYH3ixbmnSjxz4i0ob35yBhOO/y5peWY5Zb+jruRQtbawabvd+jXKTabe5KZtcVUuatJASy0nkqJJ+/A5OD8quwMdM5ETU0DUPTlWi75DtT7sBMqY5sec2+UAKxlecYxnPPyqyOhr+iVezRqG4xNPOsx2YMhi9JC8OzFtxgpwpXkkAhZAHGMngVM+fU3Isw3XGtKsTTmmbN1L6sQLBo+A/YrZJTlwPvF5TSUJKnF59c4wB8yKuqd070TbteM6Gb6SXyda17GHdQh57KXFJH4gx5dqSRk8Dg8cYO8uaY+Kkx25RpXQdj6SaMt3XaZ061L8XlJkFL1pdYcShJa2KWoOnGScDGU+oPapHqfT/s96K6k3Cw6khSfxIqXUgKcUxE8oAb8hKy4rBVk8AKFS8+O9SW/Z0qr+hfTzR+trLqSZqXVXwZ62shbCAtKcAhRLigoeZIIAwkg/bIqqa6a9m3p50o12Lq8bdd5b9seIcTKe2MuNuOO+DgIOchCE7snvUK9n7phaNTWe9641cp79HLGlZWwyraqQtCPEWnI5ACSntgkqHPBqTlkuW/rS9dyKapXR/TnSvTLrNbL3brNpdelL1AbDkZxqat1LiVZCSoK74IweOxHNc9XeBJtV1l2yYgokRXlsupPopJIP8AiK6Yckytn3GbNMWr79nfo1orqRZHZs7U9yM6Jj36BFZDfgb1LDf4igQrcG1HgcZx6c4Xs/8AS/T980jfOoWtPGesVoS4UxGVlKnlNo3rJIwcYIAAPJPpjm5vZHv2k76vVK9MaPTpvwTFD6US1PB5J8bZ+bsRhWcd8j5Vw5+WzG9fpvDHz5cWLGFqA9Divyru9nLpLA1pHu2qtSsy37JbSpKIsYkOS3UjcUjHOMEDAwSVDB4NTS0dLNP9QtOXxlnpxc9CXeCjxIDzzjpbkDBwlW8AdxzjtkGumXPjjdMzC1y7Vy9I+hiOoum0XS3a1t7EpIJkwvdi45G8ykp3EKH5gkkVTjqFNOrbWMKQopP3FdSewP21l9of/jU/UZXHC5YmEluqg9h6G6dvt/XYLR1Zsku5oKh7uiGsKUU53BOVYVjBPGeBntVLS2THlOsFW4trKM/PBxV4PdO3en/T+5dRIuorZOvdvubIt0q1Sw+01k4WFgpwVcjg/OtR7PvTKLr6Zd9R6nfeZ09aUqdlqbyFPrwVlIIHAAGTjnkfPImPJqXK3cWzfhUNK6J0ZaOkvUy16mhQtNM6UctUYvxJpuKlOLSN3mUlRwQMDPPr39a1PS/ptpq39Jrh1W13EkXGCzkQLc274aX/ADeGFLUOcFZx9ACec4rXzSe4nWqMpXRlo6f6G6sdK7tqPR1jc03f7SpYVERJU609tTvSPN23DIB4wR6isvot076Vak6RXfVlwtt4detsV5uctb48jqGA4tbKQccbvLu9e9S8+MnmHSuaKV1J010X0f6qaMvUPTum5tnudtRhEl6SpbhJSrYsnODkpOU4+3pUB6B9LLVqK33zV+sFvJ0/YgvxWWiUqfWhO5QyOQAMduTkU+fHzv6OtUzSuj+m+lumHWaBerVZ9MOaTvMFoOxXWpi3gtB4BUFcHBwDx68Gns19MNC6nnXm1aotdwfvthkluY2t/bHPnWkABOFEgoOc/Sl58ZLuejpa5wpXUnSbSnRPXk+96NgaaubM2E2pSbhIknxXAFbStODtSQSMAjGD96+umOkujGpNU3fpyNK3MXK3JdSq4SpJ8R1SFbVEbDtSQTwMYxUv6iTfi+F6Ofemel16x1tbtPiS3FakOEvvrUAGmkgqWrnjISDj61sutTGhomuXofT4vLtMdlLTji3CtLjwJC1IUeSk+Xn55xxipR0/0VpFPW24aC1QLq9/ly4cFyI6lHIUeXD3wU47Vk9SdBaVie0dH0Sy+ixWJwx0OOqczsBbBPmUe6jxk+pq/JO//Sa8KYpXUmmdN9Hb51Tn9MoujQ6wwwpLN6auLjinXEoSpXbgYyr5jKe3OK531/Y0aa1vetPtvKfRbprsZLihgqCVEAmtYcsyutJcdNHSlK6IVN+krqzdXIsV4N3EKEm3pJA8WQ2lfhI54J3qScHuARySAYRVidJXoaIMxcplt0QnvfThkKcR4bLqkrQo9j4iUJweCVgGs8n+qz23l4K13LTOmpsdiBMkzIjT0Bhw7Yw8ZZVuH6q1BbQO7KstEn0zBdUX6PM1LdJbljty1vzHnFKUXckqWSScL+tT63zGH9Q6Xktt5jTtRW99tbqNr0lxC3EvOrPJBUVNrI7AOJxnua1m3u4tzHm3W7ctaHFJUr3Bnkg8n8tc8IteXxaJ+wLb/F3+enxaJ+wLb/F3+enx+b/UW3+72f5afH5v9Rbf7vZ/lrppD4tE/YFt/i7/AD1cHXiew1056WrXaoTwcszhSlZcw3/1fAwocffNU/8AH5v9Rbf7vZ/lq4OvF1ksdOelriGoZLtmcUoLiNqAP4fYFOEj6Cuec/dj/fpZ6qn/AItE/YFt/i7/AD0+LRP2Bbf4u/z0+Pzf6i2/3ez/AC0+Pzf6i2/3ez/LXTSHxaJ+wLb/ABd/nre2O4MStJ6oYRAjQyxFjy2ywpzJcTKZQO6iPyur9PWtF8fm/wBRbf7vZ/lqQaXvDsjT2rUSmogR8JTjwojaDuMqOBylIPc1Mp4InsZ2ReIDktCmEadkQpbUy4KWSuGFKW600s5OAHDGHbcfCIGUnzU5qSa3ctQ3G4MoUhuVKceQk90hSiQD/GrblS4LipkNcNcqXdUSJbDTDGWpjaQp2MHcEEKbCEnAHmDqQe2TVGrC2rVF1U04w42Zju1bAAbUN55SBxj5Y4rPH7XJrKnnQCx3C+dW9OtwYjkhESezKklOB4bKHElSyT6CoHWRAnTYDxegzJEVwjaVsulCiPlkHtXTKWyyMz26t6d3S6aK9qTUFmvjDkS3asfeXEK8bXFBai0sfQ+dP3UKiXtC2kRLtpzo1oiM5KLL79xcYbSAVuvLWtCT9UN55/okfuoR67XV6azNeucx2UwoKZeW+pS2yDkFKicjB54o9drq9dDdXblMXPPeUp9Rd7bfz5z24+1cJw2ZTLbfbxp1R7bmnrtcrNp67QYTr8W2tvCY4nG1oLLYTn7nivbUmjNSD2PbfpxFtdXdWi3IdjDG9CPFUs5z8knmuVZN+vkphTEm83F9pX5kOSlqSfuCa+1aj1CpBQq/XQpIwUmW5gj5d6k4MpjMd+qdpu11b0L05e0eyjqS2Kt7omXZia5BZONzyXI6Utkf6RHGa8fYqsN4sNk1jLucB1lLkhuO2k43LcY8YOJA+ilAfeuVmNQX5hlDLN7ubTSAEoQiUsJSB6AA8Cvxi/32OlSGL1cmkqUVkIlLSConJPB7k+tMuDKzKb9kzk06Q9jTSuoLH1C1G9drW/EbjRjDdU5jCXtyF7PvtIP7xWDpnRXUGJ1i15dtOz12a8RVOTYkd1tKkXBl15e1JycAHbxn1x271z8zf76ypxTV6uTanVb3CmUsFasAZPPJwBz9KDUF+DjrgvVy3upShxXvS8rSDkAnPIB5FW8WVyt37JlNOz9O3246+0bqC29XNBizswIynFy32yhheAeUFXKVJ75ST9xxmG+y/HVfvZv1tpi3LbXc5C5jaGioA5eipQgn5AlJ5+hrmm66s1RdoIg3TUV2mxR2ZkTHFo/gTisWx3u8WOSZVlus22vqG0uRX1NKI+RKSKz/AI962Sr38r36F2e4dH+p9ima1eg239IIz8Ix3HQp6MnchSFrAPlCloCQfvnHpI+uOkutjvUedL0dcL2/Y5ykuRxEuJQhnKQFJIKht82T8sH+HL9ynzrlMXMuMyRMkr/O6+4VrV9yea2jOstWs2z4W1qe8twduz3dM1wN7fltzjFbvFl27b8p2mtLs6Maf1O17StvRdL2dRvWeMVT5aZSn0xtzSh4ZWo+i1YwPXP1xGvbAtNxh9Y59zkxVtw7ghsxHTjDoQ0hKsfY8VUsC43CA4tyBPlRFrGFqZeUgqH1IPNLhcrjcSg3CfKllGQgvvKXtz3xk8Vqcdmfbf1pO3jTqz2D7TcIlk1Ld5MdTUKe5GRFdURh0t+Lvx68b08/X6GvL2dkCT0x1n0gu+y26jHvCUR3lBJIdZASe/O1Q5x6EVy7Cvd5hMCPDu8+MyCSG2pK0JGfoDivJFxuCLiLimdKTNCt4kB1XiBXz3ZzmsZcFyuV37/hZnrTpb2VdJXjQN61NqzWkR+xQIUIxt0xvYHDvClFJPcDYAMZzu4rnfW92bv2sbzemmy23OmvSEIJyUhSyQP4Gvy9an1JemG494v1zuDLf5G5MpbiU/YKNaiumGFmVyvus2+NOsfZ/MPX3s133p9HfbZurCXkIQXMFW872lnjhO/KT37fUVuvZB0PqXQsbVbmprcYT0pUYMxy4lTig0HsnAPYlYAPrg1yBaLrc7PMEy03CVAkpGA7GeU2sD7pINZb+p9SPzXZr2oLq5KeADjypjhWsDsCc5OMn+NcsuC3tJfFamc8Oo/ZndkR9D6u6XTJC7Jqppx5xlKl7Vo8VpIStJBzlKsE47ZFV9+h3XlD8j9ItTXiyWyKlS5Fwl3ZQZQkDuNqiVZ9APnVIJuVwTcRcRPlCaFBQkeKrxAR67s5zWbe9U6lvbKWbzqC6XFpP5USZa3Ej9yia18NmVs+ztNNU9w8v8QOeY+cfrfXmus/YVs1zhWXUd4lxFswbiY6YjyuA94ZdC9v2JArkqthEvl7hx0x4l4uEdlGdrbUlaUp5zwAcVvm47yYdYzjdXa4tAdIeqMu8I07dLXLt+mJk5p+5pddSltxCFZ9DkqxkDHzq0egb+n40zqV0ljPpgr+KTUQQtWSplQLOBn8xTtBPOSD965R/SXUf/zBdv8AfHP/ADrBRNmIne/olvpl7/E8cOHxN3fdu75+tYy4ss5d1qZSeloNdBNfMXWSzd4rFntkbcX7nJfSGEtj9YEHJz8v44q2dIFjX3skztE2CQzJvdsQUqjJVhTmyR4qSkHkhQGAcd+K5nvOqdS3mMiLd9QXWewj8rcmWtxI/cTisG13K4WqYmZbJ0mFJT+V2O6ptY/eDmrlx5ZTzfMSZSOqPZ9996TdGtVaj1fGdtC3nsw40pGx11aWyEgIVz5lHA+xPasr2a7Fdv8Aoy6tje5L8a8e/LgIyMvhcVLaSPutJH7q5Vvl/vt9cS5erzcLktH5TKkLdKftuJxX5Fv18isIjxrzcWGUDCW25S0pSPoAcCsZcFsvnzVmenU3sWabvlkjasfutvdiNurRGbLmPM40XEuJHzwePvX10GCJWiNddJrsU26/LdklDDqgFKQ83wRzzg8n6EVywzqC/MJKWb3cmklRUQiUsAknJPB7k81jC43AXAXATpImBW4SPFV4mfnuznNMuC5W233/AATPWnS/spaSvOgb/qbVetIj1it8K3qiqXMRsSslxKipJP5gPD7jOdwxUj9kh2XetY9QtamIti23ieFxlr9T4jqyn6lIWnP3rlO86o1Jeo6I94v90uDKMbW5MpbiRjtgKNY0G83iCx4EG6zorWd2xmQtCc/PANMuC5S7vmkzkdOeyfpXUFm6uarm3S2PRY7LS4y1uYADqltuJT377CD+8fOsjofpq+W32odXzZ1udYjYku+IrGNrzuWz/wB4JJH2rl9vUF+aW4tu93JCnFbnCmUsFZwBk88nAA/dRN/vqX1vpvVyS64AFrEpYUoDsCc84q5cOVtu/cJlI6DtWkdQf9MhUly3ONsCcu5eIr8vu/KQv7FXA+prbdRtCx797Wlu/SiMpFhuaB4ClL2plLZYyWwQc/mwD8+3rXMvx2+e9GV8ZuPvBR4fi+9L37c525znGfSviTebvKeZek3Wc+4wreytyQtRbV80kng8Dt8qfDlve/rR2jumyRdb2zrMbbFtNqs+gI7BTGDDbSDIWWweMebcFbuOBhP8eO+uf+ePV3+t5H/Oa0tw1Zqm4Psvz9R3eS6wMNLdmOKU3xjyknjitS+87IeW++6t11ZKlrWoqUon1JPc1eLhuF3TLLb4pSld2Cpp0mhuyb4lyBcpVvuTchhMd1ttK2hvcCfxEnukqKB6jnkGoXUh6eSbpG1K0q0ttvPbFLU248G0qSgeITuJABGwKGfVI79qzn/rVntPdczmm75Hv8gFU6K9HlR0MvpXHgMtuJSIzXhkIxhSF8Z8qkDg7sxHW2jdQQ9ZXmMizzPAbnvJbeLJQ2tHiKCVBR42nHBzipnq/T8JmwfCYDC7XGS2JD7TivEe9/daLzcYkgYR4YSMjupIzyKhHUxxcu5W2+JWot3a2R3yc8eIhPgu/wD3GlnH1Fc+O+tLWtVpm4ttpckPW1hBx+e4MlQHzKQoq/wr7MDT0LPvt6dnuJI/Dt7BCFD1HiOYKT9kKFaLknHJNSeyaPkyJUNN5mR7OzKUnwxIV+K4D2KWxyAfRSsJ+tdL491HvYkW2eqQmDYYLUVkhS5NwffecQDwlOGikKJPbCP3gZqU66vV11DBsVtk2i0QYNmZEe3tXBzwHnkHGcpK/Ik4GMnsPzk5NY0vTlvjBaG3ZOmbizHT7woOLUw2V+Zltw8qU6sckIGB3wNqgI0rSlwjXKWL+2+IsdRRIkxlpe8IlRQHCATuQFjB7fLIJFY8W7Xy+Lwzb40lTd301ItLriB4JhSFFo8/nAc3+ID/AGVgViuWa2SCpVq1BGcH6rU1sxnT/wAyB/8AXU4Gn/AsSIl1U1DtaXGUvOrdKozwWPLKjOHJC9uCpv1HoOya/vdmdgASY7qZtucUQxLbHlVj0UP1VdspPPIPIIJ1jd/aWPZzS15Sje21FkDGf8nmsvH+CFk+vat7ZrJd7To/UD9xt0mG3PVFtiVPtqRlSnkvHAI82AxzzxkVCASOxIqe2uE0rSNmts6W7CbuMmRdZLyU7z4DCC2yoD5lfvCQOOSPpTLcIm7Rdj6Rk263XF63w3HvEYT5HpUJK30IMRgqPiJS4VIdKsgYCOSSrdS15aisXeYxBdW7FbfWlhxZBKkBRCSccZIxVwX6PNhQXrtDsqHNReC4xcpAeAjxVRkpd3NggZWUeGoAHgtYAxkVSh5OazxfdXIqT9MtLw9Yaqj6fk3oWl6WpLcVZjF4OOFQASQCNvcnP0qMVN+gn+ebSX+tGf8AmFbztmNsSe2+HSe0zNazNE2jXUWRqCO6plDEmAthl5xIJUhLgUrkYI5ABPYmq2vdsm2a7y7TcmFR5kN5TL7au6VpOCP/AO10cjSVttHVDVPVqXdU3WBp68vvv26A0oyEO7sp37sAJBIJUMjyn5GtB0utEHqFedfdUdRR7cERCt+NGmlRioecKiku7RlSEAAYxzmuOPLZ5vmfy1cVCUq8b5+jN16fXwam1Do2Vfo+x+zvWmOWVqx+dlYS0kKSR2znk962HUO52vSnTLQOoLRo/TTV1vEd1955UBKwjBRwEqyD37nJHOMZNb+X60nVz9Ut07atKJ0LeL5qG4PG4KV7rZ4MVxPiKeAClOugg4aAKR/aJIHbItrVdhsF7sHTvqTbbFaoD17uCbZcYHgZjLdUVp8QIBH9BRxn1HyycXqXZrDaPapt2nomn7X8KlvwIaoio48JKXShKlBIx5uSc/Os/L28f3wddId7P+gbB1Dv0+0Xafc4r8aGuW2IqUbVISQCCpWSDlScYB4z2qtHBtcUkehIrq3pVAgWj2s9cQLdCYiw2LQ+lthlAQhIzH4AHaoV0XvNn6h6mf6f6k0vY2oc5h73B6HCS09EcSCoYWPMeAeVEkkck5NSct3b9ai9USsmh9L3Xopf9Zxp93+LWcsNusOIbSxvcWB5cZUoY+e3mo3cLJptnptbr9G1GH78/OWzJtXh48FoAlK8+vYffeB+qc2Ro6Gbf7PXVWAV7zGuUVkqx32u4z/hXlq+zWqR7MOlb7Dsdvj3abexFW9GjgOOJS28gAnkkkoBPzPpVmd37+/4NKUpV2dXlsdJrxbdFadtlrcdjQW3rnNmQWpC5rzmSoedJ2tgDASMd/31rIl26fSur1yuFi0hNuFukQj8JtojhYE0tpwVNZwUBW7y8j/hW5ybm9eGdKmpV/6h03cbp0D1BfNY2GDbr9aJzJiOsRGmHC0spSULS2AMeY4BGaoCtYZ9tlmk16XdOrtr5u/rtqygWi3Ll/k3eM4PyMjkYKsKwf7NQquhOl6NTaEs+g59tslxks3G4qut1WzHKh7uQWG0ZHfDanl4/tpNQHrvplGhOslwjMxGXICpCZ8NpwZacZWd+wgd0g7kfZNYx5N52LZ4VzSuiuo9/wBMaAnaMuMDp9p+Q7c7ExLmocYHhqSrdlKEkEJJJOVHJICR6c6JTFv050QZ6g2O0W1y73m8uNOrdjofbtzQKylpCFggZwOSM4P2pOXcl17OqnY1ruUm3yLhHt8p6HGx476GVKbaz23KAwM/WsSuktQW2bqPTXTO12oq08NYeIzdrdDSGWHUpcQVSPDAAyUpCvttA47+94i6Y031F/R4ztBRtJQnURZlvksFcx1AAC1rcLRV4mSVDCgOAPrWfm/B1czUroHpVbdFnU3UW1x7PZ9Q2a222ZcbdKfaUpwoCRsRvOCABxkYOckGsbQMi39RumWuYF20/ZIkiyW/3+3SYUFDK2SNxKMp5IO31+ZznitXl19f2nVTTFiub2mpOokRz8NjSERlvE4HiLBISPnwk5+XHzraaKsunLra9QSL5qIWmTBgKft7Phbve3h2bz6Z4H/ez2Bq5oGsZUP2UIc/4PYpBjXoQ0MyLe240UhGd5SRgrPOVdzk1oOiDFn1LpfqVIuunbO7Ji2iVPjPCKAphakK8qB2SlJGUgDis3kurb9U6+VI0q7el6rF/wCwDWd6naTslwnWVyKmO++wStfiPjlagcnGcYGMgYORmvWYmBrr2cbxqidZ7XCvVguSGWpEGIhjxGVbBsUEgA43n09B9a18vn196OqjaVc2inZjmmdOo0RosOLbWoX24z7ey60+orHlS44DhIT6DB/41ovaW0/atNdXrpbbNERDhlDTqWGxhKCpAJAHoM5OPrVnJvLqlnjatqUpXRClKUClKUClKUClKUClKUClKUCsq0TV225xpzaEOKYcC9ixlKwO6SPUEcEVi0oLotc5u9WuROgXWTcxAhNIhWdxsl9p5DjJbKQU7VJ2sgK2nJAUcDcQNENPR5OnZlgud1abOnnfiDUlltTvjQ3tqHQhI7lDgRwSB5nMkYr70NqR2XbI8RMq3N3K3shMUTlJbQVodDjTgcJBCh5kYBA5QSCAsjM1BEuVl1DBmNxGnXLdbWnbjEJ2pdTJbU48wnblJSUl9flOAknH5cVw9XTaHOajtlpR4OlbUGHR/wC8pmHZR+qB+Rr0IwCof06jcqTIlyVyZT7jzziipbjiipSifUk1uNZ2NFnuLbsJxT9pnN+826QcEuNE9lY4C0kFKh6KB9MVoq7Y69xipLa9caihMsx3JYnRmXEONsy0+IEFKSkbCfMjynGUkEcYIIBFn9XbgjTehenlx07b4lql3OyveM9HSoLSFKSVAEk5zvVyrJ571RdXN7QH+bTpP/qRz/w6554ztj/fpqXxVRXGfMuMlUiY+p1xWAfQAAAAADgAAAcfKvW0Xe4WpxSoUgoSvAcaUAtt0A5wtBylQyOxBrBpXXTKX2RnTmqrjEtaoEm0XKS4llDsL8VhxRwBltZBSSe5CsD0SKl9mKH9ZtTIsppdiYW1aUIbGXDGZKSZO1WAloupC1EkcrV35qM6Qtsm3Wr31pKU3i7trj20LOAwxgh6So+idoUkH/TP6oqZactsq26YtLz7lv8AdEPyYN4E5aGwhMd5DiGWiojcStSlZT5SfzcJNcM61EW1xdm7daW7Em/LvlwaQth6QAsNsAlAWhBUAScNJT2xgr5O7ivqkWu765eJ7LQkIfZipUA422EIccWoqcUlIAwnJwOB5UgkAk1Ha64TUSlTTotdtO6f6g26/wCpZUxmLbXBJbRFYDinXEnypOSMD1J57Y9ciF0q5Tc0k8LntXVCz6a62XPUtrfl3PTd+W58WiPxg2Sh1aiUhO4hRSMEE4zkjjNfWjdeaE0hfNSWCIbrcNEakilt9K2QiRDX5wnaNxCwArvwe39HmlqVj4sa12qaagh9O7VEl/CL1c9QSnU7YoVE92aYyfzLJUStQHYAAZ5J9DanUmBpuf0V6WIv9/fs6k290srRCL6VjybgcKBB7Y7+vaud6neuOop1To6y6bd07BiNWVrwoTzTzhWhJxuyCcHO0d+3pUywu5ol9s7qJ1AiSrTpnS2kve2LJpz8Vl50lDsmSTuU6pIJCcEnb6jJ55wJfrDXXTi/9SbN1Pcn3lFwjKivP2dEUY8Zkgj8YnGzIHoScHtnih6VfixTtV/2jqxo2zdd9Q68aNzm2+7WtxoMiOG3EPKW35OVYxtbzu+uMVHtFaj0H06uM3VNius++3pcdxu2RnYPgoiqXxvdUSckA/q9+R61UNKnw4r2q2tOax0pF6K6o05OuVx+P3+QmS8r3QKaSpCtyU53A+YjlXpntxz9X7WWkZvs92TQ0efcBd7ZM9/JVDw2tZDmW927gDxPzY529hnio6Vfim9/nadl09QNXdP+qcW337UVwuGndTRYqI03wIXvDUwJydycKG05J7+hA5xmvLo/rjQWltQ6iYVHudug3K2+5Q7pw5KjqwdzhSOAVEg+X8u1I55NU3SnxTr1+l7edrtgar6b2vpTqbQke+X2S9dH23hcHIA2rUggjCN+UjygcknnP0qoNPt2t68Rm71Kfi28r/HdYa8RaU/RORnPbvWDStY4dd6+0t2m/WLUVrv2tXLjpubKNsVHaaYYUx4AjpbQG0tpSFEEBKEnPGSTwKlnVvVeg9YaY0jEYvFzVdLJFRBky3bcAZLWUDcfP3SAsgc5Ku4zmqcpU+OePwbWh171XpPVX6NfoxKuDiLTa0W1SJccIUUoJKV5BwSc9sDtU56dtWi09EbYudqv9DzdZjyn/f7amazcggjYUtqztSnjnAyfnxXO1T+J1EizNG2zSurdNtXqHaVK9weblKjvspV3RuAIUnt3HpWMuP8AbMYsy87qV66h3iyzrZ1Us3UCDrBu3TmWt7aS0qMoZUhstZ8rZCVDAwOTx3rE6jXTpXre/r1iq73mzS5gQu4WtEEPFToSAotubgkA47n15xzioTf9WtStPnTljssezWhcgSX0JcLz0hxIIQXHFckJCjhIAHJOMnNRarjx33S1b3SLV2gtNnV785y6QPjMB62wo7bIfDLLicb1qyNys44AA4PzGPPorq3R2ltOavg3q43FD99hqgNeBDC0towrDh8wyfN+X6d+eKlpVvFLv8p2W5pTVOi5nR65dPNR3KfbgzdDPgTWIni+KMY2qRkYJAPr69+OcvpdrDp3piBraIqXe47F9hLt0RC4yXlttlBHirIKRkqJOwdhxk96pilLxS7n/J2W1orU2hrZ0b1Ro+ddrmiffX21+IiAFIaSy6FI/XGSoJ57Y3euK+tI6u0bbug1/wBFTLjcUXW7yBJyiGFNNlBTtRncM52DJ9N3Y45qOlLxy/8Ap2XdetZdPdQaI0dHl3W82t7TsdLb9rixApEpwbSVhe4JSSQcqIJ57fOMe0JqnTutdcnUun5MxYksoQ8xIj+GWihIAwdx3Z5+WMetVxSmPFMbuFy2UpSuiFKUoFKUoFKUoFKUoFKUoFKUoFKUoMyySIUW7RpFxgJnw0LBejqWpHiI9RuSQQfkc9/nV22dmdfI1uhWK3NyYzkyMFSozy1NwkoCkNPISskoIStsLG7OGDuBSQo0NWy02/b2bo0bsZfufO73ZzapKvRX1APccE/MVjPHay6ThEe3NputruCXXdNOTH3IyW0n3q2q3bUPpQvBUkpAC0g8jGcEJNQvUun59ifbEjw34r4KosxhW5mQj5pV/wAQcEHggGrei3Bu8SLW86/ZGra1IcemOvTcq3O7QpTZXyE+XJbwknKgUkc1r4Nlk2W1i3tGI8h95bUuM5JblRlr3Et5a/OklBGFtncMHA9a5zPTWlN1IdU6wvGpLLYbTciwY9jiqjRPDb2qKCQfMfU8AfYfPJO4ftelp0mW0q2X6zyYrhbkCGgTmEKBI4BKVAEj1Ua8m9M6XLgHx69yDnBbYsh3k/LlzA+X/wCq6dp7rOqhdTDTGmEMCNdtRxnfAdwuDbsFL1xPpj1Q183PUZCcnJTt9ON2Vl2S7Y7E+FwVpDtwuoS8tpZzgJZwltJ8p5cJAx3zgVJPgahPnXeZLgGcuD4S/iN1bdLsta92xR4CdqMZAGPIU/rc5yz+lkYlutN5uky9y3YLd4uS4sZ5liJ/2dLJJSllJxjalXhHCTgpQeSOaxeqN6gOMyWbtaA5cny680h+W6tyM+6ob1cFKdiUIbbSkp5KMjy43fet7tbk2REC7KSJKI6EKTBlqJccQnagBJyG2xkk8DJJwnGCKnUpSjlRJPzJqYY7u6WvylKV2ZKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQe0B9EaYy+5GalIbWFKZdzscA9Dgg4+xFTtzXdtnx2Grtb5UiPGcQ41ES6G0BQ4IQpvbsGO2UrI9Dyc19Uj0HpdepbiGVSozDIcQ2Q7KbZU4pR4SneR+884HoSQDnKT3Vm1kdN9QwtUvXlidCiN3GTJVLQn3ZDwWkpSnaAsbztCeNqgSVc+XcpMtZhOh51c+xW+LGG55bimUOEgKdRlIW+4lPmbWOU4G3nA5qB3q/2vQkYxdMsM+/FRQiT4ZG7arzLwrzFAWnCd2SognCU+VWlhdXdWIlsrlOxnWUoLTqG2ENqWgrU4eQODuWtXbGTyCOK4XC5ecfTW5PZddbWmVdb/AA5VrEq1T5TTra21lC0raTsDmEhAUFJ3HbhPJHbFa/V2tRe4LkNyKqUSrDT8nG5lIxt8PHmBwMHctee555qWXfTunNV2t+6W4xbbIbQh1bhIZaUhailK07jtUNwUlSCfESQSC6BiqnnxnYU16I8UFxlwoUULC0kg44UOCPqK6YTGpdvGlKV1ZKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQKUpQCSTknNKUoP3JxjJxX5SlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlApSlB//2Q=="""
CSS_TAB1 = r"""
:root{--azul:#242c4f;--hdr:#242c50;--dorado:#eab000;--fondo:#f0f2f5;--superficie:#fff;--superficie-alt:#f7f8fa;--borde:#e2e8f0;--texto:#1a1f36;--texto-sec:#64748b;--radio:6px;--rojo:#dc2626;--verde:#16a34a;--naranja:#d97706;--p:var(--azul);--a:var(--dorado);--bg:var(--fondo);--card:var(--superficie);--sf:var(--superficie-alt);--bd:var(--borde);--tx:var(--texto);--tx2:var(--texto-sec);--r:var(--radio);--danger:var(--rojo);--ok:var(--verde);--warn:var(--naranja)}
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'Montserrat',sans-serif;font-size:13px;background:var(--bg);color:var(--tx);-webkit-font-smoothing:antialiased}

header{background:var(--hdr);border-bottom:3px solid var(--a);position:sticky;top:0;z-index:200;padding:0 20px;height:64px;display:flex;align-items:center;justify-content:space-between}
.hdr-left{display:flex;align-items:center;gap:14px}
.hdr-titles h1{font-family:'Lora',serif;font-size:15px;font-weight:600;color:#fff;letter-spacing:-.2px}
.hdr-titles p{font-size:9px;font-weight:400;color:rgba(255,255,255,.75);margin-top:2px;text-transform:uppercase;letter-spacing:.05em}
.btn-dl{background:var(--a);color:var(--p);font-family:inherit;font-size:12px;font-weight:600;border:none;padding:8px 16px;border-radius:var(--r);cursor:pointer}
.btn-dl:hover{background:#d4a000}

.hamburger{background:none;border:none;cursor:pointer;display:flex;flex-direction:column;gap:5px;padding:6px;border-radius:var(--r)}
.hamburger:hover{background:rgba(255,255,255,.1)}
.hamburger span{display:block;width:20px;height:1.5px;background:#fff;border-radius:2px;transition:all .25s}
.hamburger.open span:nth-child(1){transform:translateY(6.5px) rotate(45deg)}
.hamburger.open span:nth-child(2){opacity:0}
.hamburger.open span:nth-child(3){transform:translateY(-6.5px) rotate(-45deg)}

.drawer-overlay{display:none;position:fixed;inset:0;background:rgba(0,0,0,.35);z-index:300}
.drawer-overlay.open{display:block}
.drawer{position:fixed;top:0;left:0;height:100%;width:272px;background:#fff;z-index:400;transform:translateX(-100%);transition:transform .26s cubic-bezier(.4,0,.2,1);display:flex;flex-direction:column;overflow:hidden;box-shadow:4px 0 24px rgba(0,0,0,.12)}
.drawer.open{transform:translateX(0)}
.drawer-hdr{padding:18px 16px 12px;border-bottom:1px solid var(--bd);flex-shrink:0;background:var(--sf)}
.drawer-hdr h3{font-size:13px;font-weight:600;color:var(--p)}
.drawer-hdr p{font-size:10px;color:var(--tx2);margin-top:3px}
.drawer-list{flex:1;overflow-y:auto;padding:6px 8px}
.drawer-item{display:flex;align-items:center;gap:10px;padding:9px 10px;cursor:pointer;color:var(--tx2);font-size:12px;border-radius:var(--r);border-left:2px solid transparent;transition:all .15s}
.drawer-item:hover{background:#f1f5f9;color:var(--p);border-left-color:var(--a)}
.drawer-num{font-family:'Roboto Mono',monospace;font-size:10px;color:var(--a);width:20px;flex-shrink:0;font-weight:500}

main{padding:14px 20px;max-width:1700px;margin:0 auto}

.filters-card{background:var(--card);border:1px solid var(--bd);border-radius:var(--r);padding:10px 14px;margin-bottom:10px}
.filters-grid{display:flex;flex-wrap:wrap;gap:7px;align-items:flex-end}
.fg{display:flex;flex-direction:column;gap:3px}
.fg label{font-size:10px;color:var(--tx2);font-weight:500;text-transform:uppercase;letter-spacing:.04em}
.fg select,.fg input{padding:4px 8px;border:none;border-radius:var(--r);font-family:inherit;font-size:12px;color:var(--tx);background:#f1f5f9;height:28px;transition:background .15s}
.fg select:focus,.fg input:focus{outline:2px solid var(--a);outline-offset:1px;background:#e8edf5}
.btn-apply{background:var(--p);color:#fff;font-family:inherit;font-size:12px;font-weight:600;border:none;padding:0 14px;border-radius:var(--r);cursor:pointer;height:28px}
.btn-apply:hover{background:#1a2040}
.btn-clear{background:#f1f5f9;color:var(--tx);font-family:inherit;font-size:12px;border:none;padding:0 12px;border-radius:var(--r);cursor:pointer;height:28px}
.btn-clear:hover{background:#e2e8f0}

/* Multi-select Especies — estilo Excel */
.species-wrap{position:relative}
.species-btn{padding:4px 28px 4px 8px;border:1px solid var(--bd);border-radius:var(--r);font-family:inherit;font-size:12px;color:var(--tx);background:#f1f5f9;height:28px;cursor:pointer;text-align:left;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;min-width:140px;max-width:200px;appearance:none;-webkit-appearance:none}
.species-btn-arrow{position:absolute;right:8px;top:50%;transform:translateY(-50%);font-size:11px;color:var(--tx2);pointer-events:none}
.species-panel{display:none;position:absolute;top:calc(100% + 4px);left:0;background:#fff;border:1px solid #b0b8c8;border-radius:4px;box-shadow:0 4px 16px rgba(0,0,0,.14);z-index:500;width:220px;font-size:12px}
.species-panel.open{display:flex;flex-direction:column}
.species-search-row{padding:6px 8px;border-bottom:1px solid var(--bd);display:flex;align-items:center;gap:4px}
.species-search-inp{flex:1;border:1px solid var(--bd);border-radius:3px;padding:3px 6px;font-family:inherit;font-size:12px;outline:none}
.species-search-inp:focus{border-color:var(--a)}
.species-search-icon{color:var(--tx2);font-size:13px}
.species-list{max-height:200px;overflow-y:auto;padding:2px 0}
.species-option{display:flex;align-items:center;gap:7px;padding:4px 10px;cursor:pointer;color:var(--tx)}
.species-option:hover{background:#e8f0fe}
.species-option.highlighted{background:#cfe2ff}
.species-option input[type=checkbox]{width:13px;height:13px;accent-color:var(--p);cursor:pointer;flex-shrink:0}
.species-select-all{border-bottom:1px solid var(--bd);font-weight:600;background:#f7f8fa}
.species-footer{padding:6px 8px;border-top:1px solid var(--bd);display:flex;justify-content:flex-end;gap:6px}
.species-ok{background:var(--p);color:#fff;font-family:inherit;font-size:11px;font-weight:600;border:none;padding:3px 14px;border-radius:3px;cursor:pointer}
.species-ok:hover{background:#1a2040}
.species-cancel{background:#fff;color:var(--tx);font-family:inherit;font-size:11px;border:1px solid var(--bd);padding:3px 10px;border-radius:3px;cursor:pointer}
.species-cancel:hover{background:#f1f5f9}

/* Filtro por color de indicador */
.ind-color-filter{background:var(--sf);border:1px solid var(--bd);border-radius:var(--r);padding:8px 12px;margin-top:8px}
.ind-color-filter-title{font-size:10px;font-weight:600;text-transform:uppercase;letter-spacing:.04em;color:var(--tx2);margin-bottom:7px}
.ind-conditions{display:flex;flex-direction:column;gap:5px;margin-bottom:7px}
.ind-condition{display:flex;align-items:center;gap:6px;background:#fff;border:1px solid var(--bd);border-radius:var(--r);padding:4px 8px}
.ind-condition select{border:none;background:transparent;font-family:inherit;font-size:12px;color:var(--tx);cursor:pointer;padding:2px 4px;border-radius:3px}
.ind-condition select:focus{outline:1px solid var(--a)}
.ind-condition .color-pill{display:inline-block;padding:1px 8px;border-radius:3px;font-size:11px;font-weight:600}
.pill-rojo{background:#fecaca;color:#7f1d1d}
.pill-amarillo{background:#fef08a;color:#713f12}
.pill-verde{background:#bbf7d0;color:#065f46}
.pill-gris{background:#e5e7eb;color:#374151}
.btn-rm-cond{background:none;border:none;color:var(--tx2);cursor:pointer;font-size:15px;line-height:1;padding:0 2px;margin-left:auto}
.btn-rm-cond:hover{color:var(--danger)}
.btn-add-cond{background:#fff;border:1px dashed var(--bd);border-radius:var(--r);padding:4px 12px;font-family:inherit;font-size:11px;color:var(--tx2);cursor:pointer;width:100%;text-align:center}
.btn-add-cond:hover{border-color:var(--a);color:var(--p);background:#fffbeb}

.s-item{display:flex;align-items:center;gap:5px}
.s-dot{width:6px;height:6px;border-radius:50%;flex-shrink:0}
.s-label{font-size:11px;color:var(--tx2)}
.s-val{font-family:'Roboto Mono',monospace;font-size:12px;font-weight:500;color:var(--tx)}
.s-div{width:1px;height:16px;background:var(--bd)}

.table-wrap{background:var(--card);border:1px solid var(--bd);border-radius:var(--r);overflow:hidden}
.table-meta{padding:7px 14px;display:flex;justify-content:space-between;align-items:center;border-bottom:1px solid var(--bd);background:var(--sf);font-size:11px;color:var(--tx2)}
.scroll-top-bar{overflow-x:auto;overflow-y:hidden;height:10px;border-bottom:1px solid var(--bd);background:#fff}
.scroll-top-inner{height:1px}
.tbl-scroll{overflow-x:auto;overflow-y:auto;max-height:calc(100vh - 310px);min-height:280px}

table{border-collapse:collapse;font-size:12px;white-space:nowrap}
thead tr{background:var(--p);position:sticky;top:0;z-index:4}
th:nth-child(1),td:nth-child(1){position:sticky;left:0;z-index:6}
th:nth-child(2),td:nth-child(2){position:sticky;left:70px;z-index:6}
th:nth-child(1),th:nth-child(2){background:var(--p);z-index:7}
td:nth-child(1),td:nth-child(2){background:#fff}
tr:nth-child(even) td:nth-child(1),tr:nth-child(even) td:nth-child(2){background:var(--sf)}
tr:hover td:nth-child(1),tr:hover td:nth-child(2){background:#eef2fa}
td:nth-child(2){border-right:2px solid #c8d0e0}
th:nth-child(2){border-right:2px solid rgba(255,255,255,.2)}

th{padding:8px 10px;text-align:left;color:#fff;font-weight:500;font-size:11px;cursor:pointer;user-select:none;vertical-align:top}
th:hover{background:#1a2040}
.th-inner{display:flex;flex-direction:column;gap:3px;min-width:70px}
.th-label{display:flex;align-items:center;gap:3px;white-space:nowrap}
.sort-arrow{opacity:.4;font-size:10px}
th.sorted .sort-arrow{opacity:1}
.col-filter{width:100%;padding:2px 5px;border:1px solid rgba(255,255,255,.2);border-radius:3px;font-family:inherit;font-size:10px;background:rgba(255,255,255,.08);color:#fff;height:19px;cursor:pointer}
.col-filter:focus{outline:1px solid var(--a)}
.col-filter option{background:var(--p);color:#fff}

td{padding:8px 10px;border-bottom:1px solid var(--bd);vertical-align:middle}
tr:nth-child(even) td{background:#fafbfd}
tr:hover td{background:#eef2fa}

/* SEMÁFORO CELLS */
.sem-cell{display:inline-flex;align-items:center;justify-content:center;padding:4px 14px;border-radius:4px;font-size:11px;font-weight:600;letter-spacing:.03em;cursor:pointer;transition:filter .15s;user-select:none;min-width:80px}
.sem-cell:hover{filter:brightness(.92)}
.sem-verde  {background:#bbf7d0;color:#065f46}
.sem-amarillo{background:#fef08a;color:#713f12}
.sem-rojo   {background:#fecaca;color:#7f1d1d}
.sem-violeta{background:#e9d5ff;color:#4c1d95}
.sem-gris   {background:#e5e7eb;color:#374151}

.score-cell{font-family:'Roboto Mono',monospace;font-weight:600;font-size:13px}

.badge{display:inline-block;padding:2px 8px;border-radius:4px;font-size:11px;font-weight:500;letter-spacing:.01em}
.b-ok{background:#d1fae5;color:#065f46}
.b-alert{background:#fde8e8;color:#9b1c1c}
.b-warn{background:#fef3c7;color:#78350f}
.b-neutral{background:#e5e7eb;color:#374151}

.pagination{padding:8px 14px;display:flex;justify-content:center;align-items:center;gap:5px;border-top:1px solid var(--bd)}
.pg-btn{min-width:27px;height:27px;padding:0 6px;border:1px solid var(--bd);border-radius:var(--r);background:#fff;font-family:inherit;font-size:11px;cursor:pointer;display:inline-flex;align-items:center;justify-content:center;color:var(--tx2);transition:all .15s}
.pg-btn.active{background:var(--p);color:#fff;border-color:var(--p);font-weight:600}
.pg-btn:disabled{opacity:.3;cursor:not-allowed}
.pg-btn:not(.active):not(:disabled):hover{background:#f1f5f9;color:var(--p)}
.pg-info{font-size:11px;color:var(--tx2);padding:0 3px}

/* TOOLTIP/POPUP del semáforo */
.sem-popup{display:none;position:fixed;z-index:600;background:#fff;border:1px solid var(--bd);border-radius:10px;box-shadow:0 8px 32px rgba(0,0,0,.14);padding:14px 16px;min-width:240px;max-width:320px}
.sem-popup.visible{display:block}
.sem-popup-title{font-size:11px;font-weight:600;color:var(--p);margin-bottom:10px;padding-bottom:7px;border-bottom:1px solid var(--bd)}
.sem-popup-row{display:flex;align-items:center;justify-content:space-between;gap:12px;padding:4px 0}
.sem-popup-ind{font-size:11px;color:var(--tx);flex:1}
.sem-popup-close{position:absolute;top:8px;right:10px;background:none;border:none;font-size:16px;color:var(--tx2);cursor:pointer;line-height:1}

/* MODAL indicadores */
.modal-overlay{display:none;position:fixed;inset:0;background:rgba(0,0,0,.4);z-index:500;align-items:center;justify-content:center}
.modal-overlay.open{display:flex}
.modal{background:var(--card);border-radius:10px;width:94vw;max-width:1320px;height:86vh;display:flex;flex-direction:column;overflow:hidden;box-shadow:0 20px 60px rgba(0,0,0,.2)}
.modal-hdr{background:var(--p);padding:12px 18px;display:flex;align-items:center;justify-content:space-between;flex-shrink:0}
.modal-hdr h2{font-size:14px;font-weight:600;color:#fff}
.modal-hdr-right{display:flex;gap:8px;align-items:center}
.modal-search{padding:4px 10px;border:1px solid rgba(255,255,255,.25);border-radius:var(--r);font-family:inherit;font-size:12px;background:rgba(255,255,255,.1);color:#fff;width:200px}
.modal-search::placeholder{color:rgba(255,255,255,.5)}
.modal-search:focus{outline:2px solid var(--a);outline-offset:1px}
.btn-modal-dl{background:var(--a);color:var(--p);font-family:inherit;font-size:11px;font-weight:600;border:none;padding:4px 12px;border-radius:var(--r);cursor:pointer}
.btn-close{background:rgba(255,255,255,.15);border:none;color:#fff;font-size:18px;cursor:pointer;width:28px;height:28px;border-radius:50%;display:flex;align-items:center;justify-content:center}
.btn-close:hover{background:rgba(255,255,255,.25)}
.modal-body{flex:1;display:flex;flex-direction:column;overflow:hidden}
.modal-meta{padding:6px 14px;font-size:11px;color:var(--tx2);background:var(--sf);border-bottom:1px solid var(--bd);flex-shrink:0}
.modal-stb{overflow-x:auto;overflow-y:hidden;height:10px;background:#fff;border-bottom:1px solid var(--bd)}
.modal-sti{height:1px}
.modal-ts{flex:1;overflow:auto}
.modal-ts table{border-collapse:collapse;font-size:12px;white-space:nowrap}
.modal-ts thead tr{background:var(--p);position:sticky;top:0;z-index:5}
.modal-ts th{padding:8px 10px;text-align:left;color:#fff;font-weight:500;font-size:11px;cursor:pointer;user-select:none}
.modal-ts th:hover{background:#1a2040}
.sort-arrow-m{opacity:.4;font-size:10px;margin-left:3px}
.modal-ts th.sorted .sort-arrow-m{opacity:1}
.modal-ts td{padding:7px 10px;border-bottom:1px solid var(--bd);vertical-align:middle}
.modal-ts tr:nth-child(even) td{background:#fafbfd}
.modal-ts tr:hover td{background:#eef2fa}
.modal-pg{padding:7px 14px;display:flex;justify-content:center;align-items:center;gap:5px;border-top:1px solid var(--bd);flex-shrink:0}
"""
CSS_MM = r""".mm{--hdr:#242c50;--azul:#242c4f;--dorado:#eab000;--fondo:#f0f2f5;--card:#fff;--sf:#f7f8fa;
--bd:#e2e8f0;--tx:#1a1f36;--tx2:#64748b;--r:6px;--rojo:#dc2626;--verde:#16a34a;--amar:#eab000;--gris:#94a3b8;--nar:#d97706}
.mm h1,.mm h2{font-family:'Lora',serif}
.mm .mono{font-family:'Roboto Mono',monospace}
.mm .tit{margin-bottom:14px}
.mm .tit h1{font-size:22px;color:var(--azul)}
.mm .tit p{color:var(--tx2);font-size:12px;margin-top:3px}
.mm .bar{display:flex;gap:8px;margin-bottom:16px;flex-wrap:wrap;align-items:center}
.mm .bar span{color:var(--tx2);font-size:11px;text-transform:uppercase;letter-spacing:.04em;margin-right:2px}
.mm select,.mm .pbtn{background:var(--card);border:1px solid var(--bd);border-radius:var(--r);padding:6px 12px;font-size:12px;
font-family:'Montserrat';color:var(--tx);cursor:pointer}
.mm .pbtn.on{background:var(--azul);color:#fff;border-color:var(--azul)}
.mm .kpis{display:grid;gap:12px;margin-bottom:16px}
.mm .k5{grid-template-columns:repeat(5,1fr)}
.mm .k4{grid-template-columns:repeat(4,1fr)}
.mm .kpi{background:var(--card);border:1px solid var(--bd);border-radius:var(--r);padding:13px 15px}
.mm .kpi .lbl{font-size:9.5px;text-transform:uppercase;letter-spacing:.04em;color:var(--tx2);margin-bottom:6px;line-height:1.3;min-height:26px}
.mm .kpi .val{font-family:'Roboto Mono';font-size:23px;font-weight:500;color:var(--azul);line-height:1}
.mm .kpi .sub{font-size:10.5px;margin-top:5px}
.mm .up{color:var(--rojo)}
.mm .down{color:var(--verde)}
.mm .flat{color:var(--tx2)}
.mm .dots{display:flex;gap:3px;margin-top:7px}
.mm .dot{width:13px;height:13px;border-radius:3px}
.mm .v{background:var(--verde)}
.mm .a{background:var(--amar)}
.mm .r{background:var(--rojo)}
.mm .g{background:var(--gris)}
.mm .panel{background:var(--card);border:1px solid var(--bd);border-radius:var(--r);padding:16px 18px;margin-bottom:16px}
.mm .panel h2{font-size:15px;color:var(--azul)}
.mm .panel .desc{font-size:11px;color:var(--tx2);margin:2px 0 12px}
.mm .ph{display:flex;justify-content:space-between;align-items:flex-start;gap:12px;flex-wrap:wrap}
.mm .grid2{display:grid;grid-template-columns:1fr 1fr;gap:16px}
.mm .grid3{display:grid;grid-template-columns:repeat(3,1fr);gap:16px}
.mm canvas{max-height:240px}
.mm .hm{overflow-x:auto}
.mm .hmgrid{display:grid;gap:3px}
.mm .hmcell{height:20px;border-radius:2px;font-size:8px;color:#fff;display:flex;align-items:center;justify-content:center;font-family:'Roboto Mono'}
.mm .hmlbl{font-size:10px;display:flex;align-items:center;padding-right:8px;white-space:nowrap}
.mm .hmmes{font-size:8px;color:var(--tx2);text-align:center;font-family:'Roboto Mono'}
.mm .ident{background:var(--card);border:1px solid var(--bd);border-radius:var(--r);padding:16px 20px;margin-bottom:16px}
.mm .ident h1{font-size:22px;color:var(--azul);margin-bottom:3px}
.mm .ident .ruca{color:var(--tx2);font-size:12px;margin-bottom:11px}
.mm .chips{display:flex;flex-wrap:wrap;gap:7px}
.mm .chip{background:var(--sf);border:1px solid var(--bd);border-radius:20px;padding:4px 11px;font-size:10.5px}
.mm .chip b{color:var(--azul)}
.mm .chk{display:flex;flex-wrap:wrap;gap:6px}
.mm .chk label{font-size:10.5px;background:var(--sf);border:1px solid var(--bd);
border-radius:14px;padding:3px 9px;cursor:pointer;user-select:none}
.mm .chk input{margin-right:4px;vertical-align:-1px}
.mm .chk label.on{background:var(--azul);color:#fff;border-color:var(--azul)}
.mm table{width:100%;border-collapse:collapse;font-size:12px}
.mm th{background:var(--azul);color:#fff;text-align:left;padding:8px 10px;font-size:11px}
.mm td{padding:7px 10px;border-bottom:1px solid var(--bd)}
.mm tr:nth-child(even) td{background:var(--sf)}
.mm .badge{font-family:'Roboto Mono';padding:2px 8px;border-radius:10px;font-size:10px;color:#fff}
.mm .bup{background:var(--rojo)}
.mm .bdown{background:var(--verde)}
.mm .tag{font-size:10px;background:var(--sf);border:1px solid var(--bd);border-radius:10px;padding:1px 7px;color:var(--tx2)}
.mm .pill{font-family:'Roboto Mono';padding:2px 8px;border-radius:10px;font-size:10px;color:#fff}
.mm .pill.v{background:var(--verde)}
.mm .pill.a{background:var(--amar);color:#3a2e00}
.mm .pill.r{background:var(--rojo)}
.mm .pill.g{background:var(--gris)}
.mm .dgi{display:flex;justify-content:space-between;align-items:center;padding:5px 0;font-size:11px;border-bottom:1px solid var(--sf)}
.mm .dgi .nm{color:var(--tx2)}
.mm .dgcol h3{font-size:11px;color:var(--azul);font-weight:600;border-bottom:2px solid var(--bd);padding-bottom:5px;margin-bottom:6px}
.mm .leg{display:flex;gap:14px;font-size:10px;color:var(--tx2);margin-top:10px;flex-wrap:wrap}
.mm .leg span{display:flex;align-items:center;gap:5px}
.mm .leg i{width:11px;height:11px;border-radius:2px}
.mm .note{background:#fffbeb;border-left:3px solid var(--dorado);padding:9px 12px;font-size:11px;color:#78350f;border-radius:0 4px 4px 0;margin-top:12px}
@media(max-width:980px){.mm .k5,.mm .k4{grid-template-columns:repeat(2,1fr)}.mm .grid2,.mm .grid3{grid-template-columns:1fr}}.mm .wrap{max-width:1240px;margin:0 auto;padding:16px 20px 60px}
"""
BODY_TAB1 = r"""

<header>
  <div class="hdr-left">
    <button class="hamburger" id="hamburger" onclick="toggleDrawer()"><span></span><span></span><span></span></button>
    <img src="__LOGO__" height="46" alt="SAGYP">
    <div class="hdr-titles">
      <h1>Tablero de Control de Carnes</h1>
      <p>Subsecretaría de Mercados Agroalimentarios e Inserción Internacional · SAGYP</p>
    </div>
  </div>
  <button class="btn-dl" onclick="downloadExcel()">⬇ Descargar Excel</button>
</header>

<div class="drawer-overlay" id="drawer-overlay" onclick="closeDrawer()"></div>
<div class="drawer" id="drawer">
  <div class="drawer-hdr"><h3>Desglose por Indicador</h3><p>Tocá un indicador para ver el detalle</p></div>
  <div class="drawer-list" id="drawer-list"></div>
</div>

<!-- Semáforo popup -->
<div class="sem-popup" id="sem-popup">
  <button class="sem-popup-close" onclick="closePopup()">×</button>
  <div class="sem-popup-title" id="sem-popup-title"></div>
  <div id="sem-popup-body"></div>
</div>

<main>
  <div class="filters-card">
    <div class="filters-grid" id="filters-row"></div>
    <div class="ind-color-filter">
      <div class="ind-color-filter-title">Filtrar por color de indicador <span style="font-weight:400;color:var(--tx2)">(se aplican en AND)</span></div>
      <div class="ind-conditions" id="ind-conditions"></div>
      <button class="btn-add-cond" onclick="addIndCondition()">＋ Agregar condición</button>
    </div>
  </div>
  <div class="summary-bar" id="summary-bar"></div>
  <div class="table-wrap">
    <div class="table-meta"><span id="record-count">Cargando...</span></div>
    <div class="scroll-top-bar" id="scroll-top"><div class="scroll-top-inner" id="scroll-top-inner"></div></div>
    <div class="tbl-scroll" id="tbl-scroll">
      <table id="main-table">
        <thead><tr id="table-head"></tr></thead>
        <tbody id="table-body"></tbody>
      </table>
    </div>
    <div class="pagination" id="pagination"></div>
  </div>
</main>

<div class="modal-overlay" id="modal-overlay" onclick="closeModalOutside(event)">
  <div class="modal">
    <div class="modal-hdr">
      <h2 id="modal-title">—</h2>
      <div class="modal-hdr-right">
        <input class="modal-search" type="text" placeholder="Buscar..." oninput="filterModal(this.value)" id="modal-search-inp">
        <button class="btn-modal-dl" onclick="downloadModalExcel()">⬇ Excel</button>
        <button class="btn-close" onclick="closeModal()">✕</button>
      </div>
    </div>
    <div class="modal-body">
      <div class="modal-meta"><span id="modal-count">—</span></div>
      <div class="modal-stb" id="modal-stb"><div class="modal-sti" id="modal-sti"></div></div>
      <div class="modal-ts" id="modal-ts">
        <table id="modal-table"><thead><tr id="modal-thead"></tr></thead><tbody id="modal-tbody"></tbody></table>
      </div>
      <div class="modal-pg" id="modal-pg"></div>
    </div>
  </div>
</div>

"""
MACRO_MK = r""" <div class="tit"><h1>Evolución nacional del padrón</h1></div>
 <div class="bar"><span>Filtros</span>
  <select id="fTam"><option value="">Tamaño: todos</option></select>
  <select id="fEsp"><option value="">Especie: todas</option></select>
  <select id="fAct"><option value="">Actividad: todas</option></select>
  <select id="fTra"><option value="">Tránsito: todos</option></select>
  <select id="fReg"><option value="">Región: todas</option></select>
  <select id="fProv"><option value="">Provincia: todas</option></select>
  <button class="pbtn" id="fClear">Limpiar</button></div>
 <div class="kpis" id="mkpis"></div>
 <div class="grid2">
  <div class="panel"><div class="ph"><div><h2>Estado del padrón en el tiempo</h2></div>
    <select id="selEstado"></select></div><canvas id="cEstado"></canvas></div>
  <div class="panel"><h2>Score promedio vs. faena expuesta</h2>
    <canvas id="cScore"></canvas></div>
 </div>
 <div class="panel"><h2>Incumplimiento por indicador en el tiempo</h2>
  <div class="hm"><div class="hmgrid" id="mheat"></div></div>
  <div class="leg"><span><i style="background:#f1f5f9;border:1px solid #e2e8f0"></i>0%</span><span><i style="background:#fca5a5"></i>~20%</span><span><i style="background:#dc2626"></i>~35%</span><span><i style="background:#7f1d1d"></i>50%+</span></div></div>
 <div class="grid2">
  <div class="panel"><h2>Indicadores más incumplidos · mes actual</h2><canvas id="cRank" style="max-height:330px"></canvas></div>
  <div class="panel"><h2>Concentración: riesgo vs. faena</h2>
   <canvas id="cConc"></canvas>
   <div class="note"><b>Cómo leer los ejes.</b> Eje X = % de establecimientos acumulados, <b>ordenados de mayor a menor faena</b>. <b style="color:#242c4f">Faena</b>: cómo se acumulan las cabezas faenadas. <b style="color:#dc2626">Riesgo</b>: cómo se acumula el score sobre esas mismas plantas (mismo orden). Línea gris = reparto proporcional.</div></div>
 </div>
 <div class="grid3">
  <div class="panel"><h2>Por tránsito</h2><canvas id="cTra" style="max-height:180px"></canvas></div>
  <div class="panel"><h2>Por tamaño</h2><canvas id="cTam" style="max-height:180px"></canvas></div>
  <div class="panel"><h2>Por región</h2><canvas id="cReg" style="max-height:180px"></canvas></div>
 </div>
 <div class="grid2">
  <div class="panel"><h2>Mayores deterioros</h2>
   <table><thead><tr><th>Establecimiento</th><th>Tr.</th><th>Tam.</th><th>Score</th><th>Δ</th></tr></thead><tbody id="tEmp"></tbody></table></div>
  <div class="panel"><h2>Mayores mejoras</h2>
   <table><thead><tr><th>Establecimiento</th><th>Tr.</th><th>Tam.</th><th>Score</th><th>Δ</th></tr></thead><tbody id="tMej"></tbody></table></div>
 </div>
"""
PLANTA_MK = r"""
 <div class="bar"><span>Buscar</span>
  <input id="qSiocal" placeholder="Establecimiento SIOCAL" style="width:150px">
  <input id="qRazon" placeholder="Razón Social" style="width:220px">
  <input id="qCuit" placeholder="CUIT" style="width:150px">
  <select id="selPlanta" style="min-width:300px"></select></div>
 <div class="ident" id="ident"></div>
 <div class="kpis k5" id="ukpis"></div>
 <div class="grid2">
  <div class="panel"><h2>Evolución del score</h2><canvas id="uScore"></canvas></div>
  <div class="panel"><h2>Faena mensual por especie</h2><canvas id="uFaena"></canvas></div>
 </div>
 <div class="panel"><h2>Semáforo por grupo en el tiempo</h2>
  <div class="hm"><div class="hmgrid" id="uSem"></div></div>
  <div class="leg"><span><i class="v" style="width:11px;height:11px"></i>Verde</span><span><i class="a" style="width:11px;height:11px"></i>Amarillo</span><span><i class="r" style="width:11px;height:11px"></i>Rojo</span><span><i class="g" style="width:11px;height:11px"></i>Gris</span></div></div>
 <div class="panel"><div class="ph"><div><h2>Evolución de indicadores</h2></div></div>
   <div class="chk" id="uPick" style="margin-bottom:12px"></div><canvas id="uInd"></canvas></div>
 <div class="panel"><h2>Incumplimiento por indicador en el tiempo</h2>
  <div class="hm"><div class="hmgrid" id="uHeat"></div></div>
  <div class="leg"><span><i class="v" style="width:11px;height:11px"></i>Verde</span><span><i class="a" style="width:11px;height:11px"></i>Amarillo</span><span><i class="r" style="width:11px;height:11px"></i>Rojo</span><span><i class="g" style="width:11px;height:11px"></i>Gris</span></div></div>
 <div class="panel"><div class="ph"><div><h2>Score de la planta vs. promedio de su cohorte</h2></div>
   <div class="chk" id="uCoh"></div></div><canvas id="uComp"></canvas></div>
 <div class="panel"><h2>Desglose actual · 15 indicadores</h2><div class="grid3" id="uDg" style="gap:12px"></div></div>
"""
CODE_TAB1 = r"""const GROUP_MAP    = {"G1 · Ingreso / Trazabilidad": ["01 - % DTE Cerrados sobre Emitidos", "03 - % DTE con Faltante Ingresadas/Amparadas", "04 - % Cabezas con No Coincidencia de Sexo"], "G2 · Autorización de Faena": ["06 - % Autorizaciones de Emergencia", "07 - % Autorizaciones Posdatadas", "08 - % Aprobaciones Tardías (>10am)"], "G3 · Tipificación": ["10 - % Tropas con Dentición Uniforme", "11 - Tipificador No Habilitado en Planta", "12 - Tipificadores con Alerta Alta (Entropía)"], "G4 · Balance / Volumen": ["09 - % DTe con Demora de Faena", "13 - % Faenado sobre Ingresado (últimos 30 días)", "16 - % Días Faena sin Existencias"], "G5 · Capacidad Física": ["05 - % Días con Exceso Capacidad Corral", "14 - Romaneo Excede Cupo de Faena", "15 - % Días con Stock > Capacidad Cámara"]};
const IND_LABELS   = {"01 - % DTE Cerrados sobre Emitidos": "01 · DTE Cerrados", "03 - % DTE con Faltante Ingresadas/Amparadas": "03 · DTE con Faltante", "04 - % Cabezas con No Coincidencia de Sexo": "04 · Sexo DTE", "09 - % DTe con Demora de Faena": "09 · DTe con Demora", "06 - % Autorizaciones de Emergencia": "06 · Emergencias", "07 - % Autorizaciones Posdatadas": "07 · Posdatadas", "08 - % Aprobaciones Tardías (>10am)": "08 · Aprobaciones Tardías", "10 - % Tropas con Dentición Uniforme": "10 · Dentición Uniforme", "11 - Tipificador No Habilitado en Planta": "11 · Tipificador", "12 - Tipificadores con Alerta Alta (Entropía)": "12 · Entropía", "13 - % Faenado sobre Ingresado (últimos 30 días)": "13 · Faena vs Ingreso", "16 - % Días Faena sin Existencias": "16 · Faena sin Existencias", "05 - % Días con Exceso Capacidad Corral": "05 · Exceso Corral", "14 - Romaneo Excede Cupo de Faena": "14 · Romaneo vs Cupo", "15 - % Días con Stock > Capacidad Cámara": "15 · Sobrestock Cámara"};
const IND_TO_SHEET = {"01 - % DTE Cerrados sobre Emitidos": "Ind01_DTE_Cerr_Amp", "03 - % DTE con Faltante Ingresadas/Amparadas": "Ind03_Faltante", "04 - % Cabezas con No Coincidencia de Sexo": "Ind04_Coincidencia_Sexo", "06 - % Autorizaciones de Emergencia": "Ind06_Emergencia", "07 - % Autorizaciones Posdatadas": "Ind07_Posdatadas", "08 - % Aprobaciones Tardías (>10am)": "Ind08_AproTardia", "10 - % Tropas con Dentición Uniforme": "Ind10_Denticion_Unif", "11 - Tipificador No Habilitado en Planta": "Ind11_Tipif_Habilitado", "12 - Tipificadores con Alerta Alta (Entropía)": "Ind12_Entropia_Tipif", "09 - % DTe con Demora de Faena": "Ind09_Demora", "13 - % Faenado sobre Ingresado (últimos 30 días)": "Ind13_Faena_vs_Ingreso", "16 - % Días Faena sin Existencias": "Ind16_Faena_vs_Existencias", "05 - % Días con Exceso Capacidad Corral": "Ind05_Corral_vs_Ingresos", "14 - Romaneo Excede Cupo de Faena": "Ind14_Romaneo_vs_Cupo", "15 - % Días con Stock > Capacidad Cámara": "Ind15_Stock_vs_Camara"};
const GROUP_COLS   = ["G1 · Ingreso / Trazabilidad", "G2 · Autorización de Faena", "G3 · Tipificación", "G4 · Balance / Volumen", "G5 · Capacidad Física"];
const IND_FILTER_DEFS = [{"col": "01 - % DTE Cerrados sobre Emitidos", "label": "01 · DTE Cerrados"}, {"col": "03 - % DTE con Faltante Ingresadas/Amparadas", "label": "03 · DTE con Faltante"}, {"col": "04 - % Cabezas con No Coincidencia de Sexo", "label": "04 · Sexo DTE"}, {"col": "06 - % Autorizaciones de Emergencia", "label": "06 · Emergencias"}, {"col": "07 - % Autorizaciones Posdatadas", "label": "07 · Posdatadas"}, {"col": "08 - % Aprobaciones Tardías (>10am)", "label": "08 · Aprobaciones Tardías"}, {"col": "09 - % DTe con Demora de Faena", "label": "09 · DTe con Demora"}, {"col": "10 - % Tropas con Dentición Uniforme", "label": "10 · Dentición Uniforme"}, {"col": "11 - Tipificador No Habilitado en Planta", "label": "11 · Tipificador"}, {"col": "12 - Tipificadores con Alerta Alta (Entropía)", "label": "12 · Entropía"}, {"col": "13 - % Faenado sobre Ingresado (últimos 30 días)", "label": "13 · Faena vs Ingreso"}, {"col": "14 - Romaneo Excede Cupo de Faena", "label": "14 · Romaneo vs Cupo"}, {"col": "15 - % Días con Stock > Capacidad Cámara", "label": "15 · Sobrestock Cámara"}, {"col": "16 - % Días Faena sin Existencias", "label": "16 · Faena sin Existencias"}, {"col": "05 - % Días con Exceso Capacidad Corral", "label": "05 · Exceso Corral"}];

const PS = 20;
let filteredData = [...ALL_DATA];
let sortCol = null, sortDir = 1, currentPage = 1;
let colFilters = {};
let selectedEspecies = new Set();
let indConditions = []; // [{col, color}]

// ── FILTRO POR COLOR DE INDICADOR ─────────────────────────────────────
function addIndCondition() {
  const id = Date.now();
  indConditions.push({id, col: IND_FILTER_DEFS[0].col, color: 'ROJO'});
  renderIndConditions();
}

function removeIndCondition(id) {
  indConditions = indConditions.filter(c => c.id !== id);
  renderIndConditions();
  applyFilters();
}

function renderIndConditions() {
  const container = document.getElementById('ind-conditions');
  container.innerHTML = '';
  indConditions.forEach(cond => {
    const row = document.createElement('div');
    row.className = 'ind-condition';

    // Indicador select
    const selInd = document.createElement('select');
    selInd.title = 'Indicador';
    selInd.innerHTML = IND_FILTER_DEFS.map(d =>
      `<option value="${d.col}" ${d.col === cond.col ? 'selected' : ''}>${d.label}</option>`
    ).join('');
    selInd.onchange = () => { cond.col = selInd.value; applyFilters(); };

    // Operador label
    const op = document.createElement('span');
    op.textContent = 'es';
    op.style.cssText = 'font-size:11px;color:var(--tx2);white-space:nowrap';

    // Color select con pills
    const selColor = document.createElement('select');
    selColor.title = 'Color';
    const colorOpts = [
      {v:'ROJO',    label:'🔴 Rojo'},
      {v:'AMARILLO',label:'🟡 Amarillo'},
      {v:'VERDE',   label:'🟢 Verde'},
      {v:'GRIS',    label:'⚫ Gris'},
    ];
    selColor.innerHTML = colorOpts.map(o =>
      `<option value="${o.v}" ${o.v === cond.color ? 'selected' : ''}>${o.label}</option>`
    ).join('');
    selColor.onchange = () => { cond.color = selColor.value; applyFilters(); };

    // Botón eliminar
    const rm = document.createElement('button');
    rm.className = 'btn-rm-cond';
    rm.textContent = '×';
    rm.title = 'Eliminar condición';
    rm.onclick = () => removeIndCondition(cond.id);

    row.appendChild(selInd);
    row.appendChild(op);
    row.appendChild(selColor);
    row.appendChild(rm);
    container.appendChild(row);
  });
}

// ── DRAWER ────────────────────────────────────────────────────────────
function buildDrawer() {
  const list = document.getElementById('drawer-list');
  SHEET_ORDER.forEach(key => {
    const ind = ALL_IND[key];
    const d = document.createElement('div');
    d.className = 'drawer-item';
    d.innerHTML = `<span class="drawer-num">${ind.num}</span><span>${ind.name}</span>`;
    d.onclick = () => { openModal(key); closeDrawer(); };
    list.appendChild(d);
  });
}
function toggleDrawer() {
  const open = document.getElementById('drawer').classList.toggle('open');
  document.getElementById('drawer-overlay').classList.toggle('open', open);
  document.getElementById('hamburger').classList.toggle('open', open);
}
function closeDrawer() {
  document.getElementById('drawer').classList.remove('open');
  document.getElementById('drawer-overlay').classList.remove('open');
  document.getElementById('hamburger').classList.remove('open');
}

// ── SEMÁFORO POPUP ────────────────────────────────────────────────────
function openPopup(e, est, groupCol) {
  e.stopPropagation();
  const popup = document.getElementById('sem-popup');
  const desglose = DES_DATA[String(est)];
  const inds = GROUP_MAP[groupCol] || [];
  const groupColor = e.currentTarget.dataset.color;

  document.getElementById('sem-popup-title').textContent = groupCol.split('·')[1]?.trim() || groupCol;

  const body = document.getElementById('sem-popup-body');
  body.innerHTML = '';

  if (!desglose) {
    body.innerHTML = '<div style="font-size:11px;color:var(--tx2)">Sin datos de desglose</div>';
  } else {
    // Footer hint
    const hint = document.createElement('div');
    hint.style.cssText = 'font-size:10px;color:var(--tx2);margin-bottom:8px;font-style:italic';
    hint.textContent = 'Tocá un color para ver el desglose del indicador';
    body.appendChild(hint);

    inds.forEach(ind => {
      const val = desglose[ind] || 'GRIS';
      const label = IND_LABELS[ind] || ind;
      const sheetKey = IND_TO_SHEET[ind];
      const row = document.createElement('div');
      row.className = 'sem-popup-row';
      const cls = val === 'VERDE' ? 'sem-verde' : val === 'AMARILLO' ? 'sem-amarillo' : val === 'ROJO' ? 'sem-rojo' : val === 'VIOLETA' ? 'sem-violeta' : 'sem-gris';

      const badge = document.createElement('span');
      badge.className = `sem-cell ${cls}`;
      badge.textContent = val;
      badge.style.cssText = 'min-width:70px;font-size:10px;padding:2px 8px;';

      if (sheetKey && ALL_IND[sheetKey]) {
        badge.style.cursor = 'pointer';
        badge.title = 'Ver desglose filtrado';
        badge.onclick = (ev) => {
          ev.stopPropagation();
          closePopup();
          openModalFiltered(sheetKey, String(est));
        };
      } else {
        badge.style.cursor = 'default';
      }

      row.innerHTML = `<span class="sem-popup-ind">${label}</span>`;
      row.appendChild(badge);
      body.appendChild(row);
    });
  }

  // Position near click
  const rect = e.currentTarget.getBoundingClientRect();
  const pw = 300, ph = 220;
  let left = rect.left + rect.width / 2 - pw / 2;
  let top  = rect.bottom + 8;
  if (left + pw > window.innerWidth - 12) left = window.innerWidth - pw - 12;
  if (left < 8) left = 8;
  if (top + ph > window.innerHeight - 12) top = rect.top - ph - 8;

  popup.style.left = left + 'px';
  popup.style.top  = top  + 'px';
  popup.classList.add('visible');
}
function closePopup() { document.getElementById('sem-popup').classList.remove('visible'); }
document.addEventListener('click', e => {
  const popup = document.getElementById('sem-popup');
  if (!popup.contains(e.target)) closePopup();
});

// ── FILTROS ───────────────────────────────────────────────────────────
function buildFilters() {
  const row = document.getElementById('filters-row');

  // Búsqueda
  const sg = document.createElement('div'); sg.className = 'fg';
  sg.innerHTML = '<label>Buscar</label><input type="text" id="s-inp" placeholder="Razón Social / Establecimiento / CUIT" oninput="applyFilters()" style="width:220px">';
  row.appendChild(sg);

  // Dropdowns simples
  Object.entries(FILTROS_OPT).forEach(([col, vals]) => {
    const d = document.createElement('div'); d.className = 'fg';
    const id = 'f_' + col.replace(/[^a-zA-Z0-9]/g, '_');
    d.innerHTML = `<label>${col}</label><select id="${id}" onchange="applyFilters()"><option value="">Todos</option>${vals.map(v => `<option value="${v}">${v}</option>`).join('')}</select>`;
    row.appendChild(d);
  });

  // Especies multi-select estilo Excel
  const eg = document.createElement('div'); eg.className = 'fg species-wrap'; eg.id = 'species-wrap';
  eg.innerHTML = `<label>Especies</label>
    <div style="position:relative;display:inline-block">
      <button class="species-btn" id="species-btn" onclick="toggleEspeciesPanel(event)" type="button">Todas</button>
      <span class="species-btn-arrow">▾</span>
    </div>
    <div class="species-panel" id="species-panel">
      <div class="species-search-row">
        <span class="species-search-icon">🔍</span>
        <input class="species-search-inp" id="species-search" type="text" placeholder="Buscar..." oninput="filterEspeciesPanel(this.value)">
      </div>
      <div class="species-list" id="species-list">
        <div class="species-option species-select-all" onclick="toggleAllEspeciesPending(event)">
          <input type="checkbox" id="esp-all" checked> (Seleccionar todo)
        </div>
        ${ESPECIES_ALL.map(sp => `
          <div class="species-option" onclick="toggleEspeciePending(event,'${sp.replace(/'/g,"\'")}')">
            <input type="checkbox" class="esp-cb" data-esp="${sp}" checked> ${sp}
          </div>`).join('')}
      </div>
      <div class="species-footer">
        <button class="species-cancel" onclick="cancelEspecies(event)">Cancelar</button>
        <button class="species-ok" onclick="acceptEspecies(event)">ACEPTAR</button>
      </div>
    </div>`;
  row.appendChild(eg);

  // Año
  const yg = document.createElement('div'); yg.className = 'fg';
  yg.innerHTML = `<label>Año Insp.</label><select id="f_year" onchange="applyFilters()"><option value="">Todos</option>${YEARS.map(y => `<option value="${y}">${y}</option>`).join('')}</select>`;
  row.appendChild(yg);

  // Mes
  const mg = document.createElement('div'); mg.className = 'fg';
  mg.innerHTML = `<label>Mes Insp.</label><select id="f_month" onchange="applyFilters()"><option value="">Todos</option>${Object.entries(MONTHS).map(([k,v]) => `<option value="${k}">${v}</option>`).join('')}</select>`;
  row.appendChild(mg);

  // Filtro semáforo por color de grupo
  const gg = document.createElement('div'); gg.className = 'fg';
  gg.innerHTML = `<label>Color en Grupo</label>
    <select id="f_grupo" onchange="applyFilters()" style="width:110px"><option value="">Todos</option>
      ${['G1','G2','G3','G4','G5'].map(g => `<option value="${g}">${g}</option>`).join('')}
    </select>`;
  row.appendChild(gg);
  const cg = document.createElement('div'); cg.className = 'fg';
  cg.innerHTML = `<label>Color</label><select id="f_color" onchange="applyFilters()"><option value="">Todos</option>
    <option value="ROJO">Rojo</option><option value="AMARILLO">Amarillo</option><option value="VERDE">Verde</option><option value="VIOLETA">Violeta</option></select>`;
  row.appendChild(cg);

  // Botones
  const ag = document.createElement('div'); ag.className = 'fg'; ag.style.marginLeft = '4px';
  ag.innerHTML = '<label>&nbsp;</label><div style="display:flex;gap:6px"><button class="btn-apply" onclick="applyFilters()">Aplicar</button><button class="btn-clear" onclick="clearFilters()">Limpiar</button></div>';
  row.appendChild(ag);

  // Init
  ESPECIES_ALL.forEach(sp => selectedEspecies.add(sp));
  pendingEspecies = new Set(selectedEspecies);

  // Cerrar panel al click afuera
  document.addEventListener('click', e => {
    if (!document.getElementById('species-wrap')?.contains(e.target)) {
      closeEspeciesPanel();
    }
  });
}

let pendingEspecies = new Set();

function toggleEspeciesPanel(e) {
  e.stopPropagation();
  const panel = document.getElementById('species-panel');
  const isOpen = panel.classList.contains('open');
  if (!isOpen) {
    // Sync pending to current before opening
    pendingEspecies = new Set(selectedEspecies);
    syncEspeciesCheckboxes();
    document.getElementById('species-search').value = '';
    filterEspeciesPanel('');
  }
  panel.classList.toggle('open', !isOpen);
}

function closeEspeciesPanel() {
  document.getElementById('species-panel')?.classList.remove('open');
}

function filterEspeciesPanel(q) {
  const s = q.toLowerCase().trim();
  document.querySelectorAll('.esp-cb').forEach(cb => {
    const sp = cb.dataset.esp;
    const row = cb.closest('.species-option');
    row.style.display = (!s || sp.toLowerCase().includes(s)) ? '' : 'none';
  });
  // Update select-all visibility
  const allVisible = [...document.querySelectorAll('.esp-cb')].filter(c => c.closest('.species-option').style.display !== 'none');
  const allChecked = allVisible.every(c => pendingEspecies.has(c.dataset.esp));
  const someChecked = allVisible.some(c => pendingEspecies.has(c.dataset.esp));
  const allCb = document.getElementById('esp-all');
  if (allCb) { allCb.checked = allChecked; allCb.indeterminate = !allChecked && someChecked; }
}

function toggleEspeciePending(e, sp) {
  e.stopPropagation();
  const cb = e.currentTarget.querySelector('input');
  if (pendingEspecies.has(sp)) { pendingEspecies.delete(sp); cb.checked = false; }
  else { pendingEspecies.add(sp); cb.checked = true; }
  updateAllEspeciesCheckbox();
}

function toggleAllEspeciesPending(e) {
  e.stopPropagation();
  const visible = [...document.querySelectorAll('.esp-cb')].filter(c => c.closest('.species-option').style.display !== 'none');
  const allChecked = visible.every(c => pendingEspecies.has(c.dataset.esp));
  visible.forEach(cb => {
    if (allChecked) { pendingEspecies.delete(cb.dataset.esp); cb.checked = false; }
    else { pendingEspecies.add(cb.dataset.esp); cb.checked = true; }
  });
  updateAllEspeciesCheckbox();
}

function updateAllEspeciesCheckbox() {
  const visible = [...document.querySelectorAll('.esp-cb')].filter(c => c.closest('.species-option').style.display !== 'none');
  const allChecked = visible.every(c => pendingEspecies.has(c.dataset.esp));
  const someChecked = visible.some(c => pendingEspecies.has(c.dataset.esp));
  const allCb = document.getElementById('esp-all');
  if (allCb) { allCb.checked = allChecked; allCb.indeterminate = !allChecked && someChecked; }
}

function syncEspeciesCheckboxes() {
  document.querySelectorAll('.esp-cb').forEach(cb => { cb.checked = pendingEspecies.has(cb.dataset.esp); });
  updateAllEspeciesCheckbox();
}

function acceptEspecies(e) {
  e.stopPropagation();
  selectedEspecies = new Set(pendingEspecies);
  updateEspeciesBtn();
  closeEspeciesPanel();
  applyFilters();
}

function cancelEspecies(e) {
  e.stopPropagation();
  pendingEspecies = new Set(selectedEspecies);
  syncEspeciesCheckboxes();
  closeEspeciesPanel();
}

function updateEspeciesBtn() {
  const btn = document.getElementById('species-btn');
  if (!btn) return;
  if (selectedEspecies.size === ESPECIES_ALL.length || selectedEspecies.size === 0) btn.textContent = 'Todas';
  else if (selectedEspecies.size === 1) btn.textContent = [...selectedEspecies][0];
  else btn.textContent = `${selectedEspecies.size} seleccionadas`;
}

function applyFilters() {
  const sv    = (document.getElementById('s-inp')?.value || '').toLowerCase().trim();
  const fYear = document.getElementById('f_year')?.value || '';
  const fMonth= document.getElementById('f_month')?.value || '';
  const fGrupo= document.getElementById('f_grupo')?.value || '';
  const fColor= document.getElementById('f_color')?.value || '';
  const dropVals = {};
  Object.keys(FILTROS_OPT).forEach(col => {
    const el = document.getElementById('f_' + col.replace(/[^a-zA-Z0-9]/g, '_'));
    if (el && el.value) dropVals[col] = el.value;
  });

  filteredData = ALL_DATA.filter(row => {
    if (sv) {
      const rs  = String(row['Razón Social'] || '').toLowerCase();
      const est = String(row['Establecimiento SIOCAL'] || '');
      const cuit= String(row['CUIT'] || '');
      if (!rs.includes(sv) && !est.includes(sv) && !cuit.includes(sv)) return false;
    }
    for (const [col, val] of Object.entries(dropVals)) {
      if (String(row[col]) !== val) return false;
    }
    // Especies multi-select
    if (selectedEspecies.size > 0 && selectedEspecies.size < ESPECIES_ALL.length) {
      const rowEsp = String(row['Especies'] || '').split(',').map(s => s.trim());
      if (!rowEsp.some(sp => selectedEspecies.has(sp))) return false;
    }
    if (fYear  && row['Última Inspección'] && !row['Última Inspección'].startsWith(fYear)) return false;
    if (fMonth && row['Última Inspección']) {
      const m = parseInt(row['Última Inspección'].split('-')[1], 10);
      if (m !== parseInt(fMonth, 10)) return false;
    }
    if (fGrupo && fColor) {
      const fullCol = GROUP_COLS.find(g => g.startsWith(fGrupo));
      if (fullCol && String(row[fullCol]) !== fColor) return false;
    }
    // Column-level filters
    for (const [col, val] of Object.entries(colFilters)) {
      if (val === '__nonempty__') { if (String(row[col] || '').trim() === '') return false; }
      else if (val !== '' && String(row[col]) !== val) return false;
    }
    // Filtro por color de indicador (AND entre condiciones)
    if (indConditions.length > 0) {
      const est = String(row['Establecimiento SIOCAL']);
      const des = DES_DATA[est];
      if (!des) return false;
      for (const cond of indConditions) {
        if (String(des[cond.col] || 'GRIS') !== cond.color) return false;
      }
    }
    return true;
  });

  currentPage = 1;
  renderTable();
  updateSummary();
}

function clearFilters() {
  document.getElementById('s-inp').value = '';
  document.getElementById('f_year').value = '';
  document.getElementById('f_month').value = '';
  document.getElementById('f_grupo').value = '';
  document.getElementById('f_color').value = '';
  Object.keys(FILTROS_OPT).forEach(col => {
    const el = document.getElementById('f_' + col.replace(/[^a-zA-Z0-9]/g, '_'));
    if (el) el.value = '';
  });
  ESPECIES_ALL.forEach(sp => selectedEspecies.add(sp));
  pendingEspecies = new Set(selectedEspecies);
  document.querySelectorAll('.esp-cb').forEach(c => c.checked = true);
  const allCb = document.getElementById('esp-all');
  if (allCb) { allCb.checked = true; allCb.indeterminate = false; }
  updateEspeciesBtn();
  colFilters = {};
  document.querySelectorAll('.col-filter').forEach(el => el.value = '');
  indConditions = [];
  renderIndConditions();
  filteredData = [...ALL_DATA]; currentPage = 1;
  renderTable(); updateSummary();
}

function updateSummary() {
  const tot = filteredData.length;
  const rojos = filteredData.filter(r => GROUP_COLS.some(g => String(r[g]) === 'ROJO')).length;
  const amarillos = filteredData.filter(r => GROUP_COLS.some(g => String(r[g]) === 'AMARILLO') && !GROUP_COLS.some(g => String(r[g]) === 'ROJO')).length;
  document.getElementById('summary-bar').innerHTML = `
    <div class="s-item"><div class="s-dot" style="background:var(--p)"></div><span class="s-label">Establecimientos:</span><span class="s-val">${tot.toLocaleString('es-AR')}</span></div>
    <div class="s-div"></div>
    <div class="s-item"><div class="s-dot" style="background:#dc2626"></div><span class="s-label">Con algún ROJO:</span><span class="s-val">${rojos}</span></div>
    <div class="s-div"></div>
    <div class="s-item"><div class="s-dot" style="background:#ca8a04"></div><span class="s-label">Solo AMARILLO (sin rojo):</span><span class="s-val">${amarillos}</span></div>`;
}

// ── TABLA ─────────────────────────────────────────────────────────────
function semColor(v) {
  return v === 'VERDE' ? 'sem-verde' : v === 'AMARILLO' ? 'sem-amarillo' : v === 'ROJO' ? 'sem-rojo' : v === 'VIOLETA' ? 'sem-violeta' : 'sem-gris';
}

function scoreColor(v) {
  const n = parseInt(v);
  if (isNaN(n) || v === '') return 'var(--tx)';
  if (n <= 4)  return 'var(--ok)';
  if (n <= 9)  return 'var(--warn)';
  return 'var(--danger)';
}

function buildHead() {
  const tr = document.getElementById('table-head');
  SHOW_COLS.forEach(col => {
    const th = document.createElement('th');
    const inner = document.createElement('div'); inner.className = 'th-inner';
    const lbl = document.createElement('div'); lbl.className = 'th-label';
    lbl.innerHTML = `<span>${col}</span><span class="sort-arrow">↕</span>`;
    lbl.onclick = () => sortBy(col);
    inner.appendChild(lbl);

    // Col filter dropdown
    if (!GROUP_COLS.includes(col)) {
      const sel = document.createElement('select'); sel.className = 'col-filter'; sel.dataset.col = col;
      sel.onclick = e => e.stopPropagation();
      const uniq = [...new Set(ALL_DATA.map(r => String(r[col] ?? '')))].filter(v => v).sort((a, b) => {
        const an = parseFloat(a), bn = parseFloat(b);
        if (!isNaN(an) && !isNaN(bn)) return an - bn;
        return a.localeCompare(b, 'es');
      });
      sel.innerHTML = `<option value="">Todos</option><option value="__nonempty__">— Sin vacíos —</option>` + uniq.map(v => `<option value="${v}">${v}</option>`).join('');
      sel.onchange = function() {
        if (this.value === '') delete colFilters[col]; else colFilters[col] = this.value;
        applyFilters();
      };
      inner.appendChild(sel);
    }
    th.appendChild(inner); tr.appendChild(th);
  });
}

function sortBy(col) {
  if (sortCol === col) sortDir *= -1; else { sortCol = col; sortDir = 1; }
  const semOrder = { 'ROJO': 3, 'VIOLETA': 2, 'AMARILLO': 1, 'VERDE': 0, '': -1 };
  filteredData.sort((a, b) => {
    const av = a[col], bv = b[col];
    if (GROUP_COLS.includes(col)) {
      return ((semOrder[av] ?? -1) - (semOrder[bv] ?? -1)) * sortDir;
    }
    const an = parseFloat(av), bn = parseFloat(bv);
    if (!isNaN(an) && !isNaN(bn) && av !== '' && bv !== '') return (an - bn) * sortDir;
    return String(av).localeCompare(String(bv), 'es') * sortDir;
  });
  document.querySelectorAll('#table-head th').forEach(th => {
    th.classList.remove('sorted');
    const a = th.querySelector('.sort-arrow'); if (a) a.textContent = '↕';
  });
  const idx = SHOW_COLS.indexOf(col), ths = document.querySelectorAll('#table-head th');
  if (ths[idx]) { ths[idx].classList.add('sorted'); const a = ths[idx].querySelector('.sort-arrow'); if (a) a.textContent = sortDir === 1 ? '↑' : '↓'; }
  renderTable();
}

function renderTable() {
  const tbody = document.getElementById('table-body'); tbody.innerHTML = '';
  const s = (currentPage - 1) * PS, e = s + PS;
  filteredData.slice(s, e).forEach(row => {
    const tr = document.createElement('tr');
    SHOW_COLS.forEach(col => {
      const td = document.createElement('td');
      const v = String(row[col] ?? '');
      if (GROUP_COLS.includes(col)) {
        const cls = semColor(v);
        const cell = document.createElement('span');
        cell.className = `sem-cell ${cls}`;
        cell.textContent = v || '—';
        cell.dataset.color = v;
        cell.onclick = (e) => openPopup(e, row['Establecimiento SIOCAL'], col);
        td.appendChild(cell);
      } else if (col === 'SCORE') {
        td.innerHTML = v !== '' ? `<span class="score-cell" style="color:${scoreColor(v)}">${v}</span>` : '—';
      } else if (col === 'Total Grises') {
        const n = parseInt(v);
        const color = isNaN(n) || v === '' ? 'var(--tx2)' : n === 0 ? 'var(--ok)' : n <= 3 ? 'var(--warn)' : 'var(--danger)';
        td.innerHTML = v !== '' ? `<span style="font-family:'Roboto Mono',monospace;font-weight:500;color:${color}">${v}</span>` : '—';
      } else if (col === 'Estado') {
        td.innerHTML = `<span class="badge ${v === 'vigente' ? 'b-ok' : 'b-alert'}">${v}</span>`;
      } else if (col === 'Scoring Actual' && v !== '') {
        const sc = parseFloat(v);
        const color = sc >= 3 ? 'var(--danger)' : sc >= 2 ? 'var(--warn)' : 'var(--ok)';
        td.innerHTML = `<span style="font-family:'Roboto Mono',monospace;font-weight:500;color:${color}">${v}</span>`;
      } else if (col === 'Razón Social') {
        td.innerHTML = `<a class="lnk-planta" data-ruca="${row['Establecimiento SIOCAL']}" title="Ver ficha de planta">${v}</a>`;
      } else {
        td.textContent = v;
      }
      tr.appendChild(td);
    });
    tbody.appendChild(tr);
  });
  const tot = filteredData.length;
  document.getElementById('record-count').textContent = `Mostrando ${Math.min(s + 1, tot).toLocaleString('es-AR')}–${Math.min(e, tot).toLocaleString('es-AR')} de ${tot.toLocaleString('es-AR')} registros`;
  document.getElementById('scroll-top-inner').style.width = document.getElementById('main-table').scrollWidth + 'px';
  renderPag();
}

function renderPag() {
  const tot = filteredData.length, tp = Math.ceil(tot / PS);
  const pg = document.getElementById('pagination'); pg.innerHTML = '';
  if (tp <= 1) return;
  const add = (lbl, p, active, dis) => {
    const b = document.createElement('button'); b.className = 'pg-btn' + (active ? ' active' : '');
    b.textContent = lbl; b.disabled = dis;
    if (!dis) b.onclick = () => { currentPage = p; renderTable(); };
    pg.appendChild(b);
  };
  add('‹', currentPage - 1, false, currentPage === 1);
  const s = Math.max(1, currentPage - 2), e = Math.min(tp, currentPage + 2);
  if (s > 1) { add('1', 1, false, false); if (s > 2) { const sp = document.createElement('span'); sp.className = 'pg-info'; sp.textContent = '…'; pg.appendChild(sp); } }
  for (let i = s; i <= e; i++) add(i, i, i === currentPage, false);
  if (e < tp) { if (e < tp - 1) { const sp = document.createElement('span'); sp.className = 'pg-info'; sp.textContent = '…'; pg.appendChild(sp); } add(tp, tp, false, false); }
  add('›', currentPage + 1, false, currentPage === tp);
}

function initScrollSync() {
  const top = document.getElementById('scroll-top'), body = document.getElementById('tbl-scroll');
  top.addEventListener('scroll', () => { body.scrollLeft = top.scrollLeft; });
  body.addEventListener('scroll', () => { top.scrollLeft = body.scrollLeft; });
}

function downloadExcel() {
  const wb = XLSX.utils.book_new(), ws = XLSX.utils.json_to_sheet(filteredData);
  XLSX.utils.book_append_sheet(wb, ws, 'Semáforo');
  XLSX.writeFile(wb, 'tablero_control_carnes.xlsx');
}

// ── MODAL INDICADORES ─────────────────────────────────────────────────
let modalKey = null, modalFiltered = [], modalSortCol = null, modalSortDir = 1, modalPage = 1;

// Abre el modal de un indicador pre-filtrado por establecimiento
function openModalFiltered(key, estId) {
  const ind = ALL_IND[key];
  // Buscar la columna que corresponde al establecimiento en esa hoja
  const estCol = ind.cols.find(c => c.includes('ruca') || c.includes('establecimiento') || c.includes('siocal')) || ind.cols[0];
  const filtered = ind.data.filter(r => String(r[estCol]) === String(estId));
  openModal(key, filtered);
}

function openModal(key, preFiltered) {
  const ind = ALL_IND[key];
  modalKey = key;
  modalFiltered = preFiltered ? [...preFiltered] : [...ind.data];
  modalSortCol = null; modalSortDir = 1; modalPage = 1;
  const titleSuffix = preFiltered ? ` — Est. filtrado` : '';
  document.getElementById('modal-title').textContent = ind.num + ' · ' + ind.name + titleSuffix;
  document.getElementById('modal-search-inp').value = '';
  const tr = document.getElementById('modal-thead'); tr.innerHTML = '';
  ind.cols.forEach(col => {
    const th = document.createElement('th');
    th.innerHTML = col + ' <span class="sort-arrow-m">↕</span>';
    th.onclick = () => sortModal(col); tr.appendChild(th);
  });
  document.getElementById('modal-overlay').classList.add('open');
  renderModal();
  const stb = document.getElementById('modal-stb'), ts = document.getElementById('modal-ts');
  stb.onscroll = () => { ts.scrollLeft = stb.scrollLeft; };
  ts.onscroll  = () => { stb.scrollLeft = ts.scrollLeft; };
}
function closeModal() { document.getElementById('modal-overlay').classList.remove('open'); }
function closeModalOutside(e) { if (e.target === document.getElementById('modal-overlay')) closeModal(); }
function filterModal(q) {
  const base = modalKey ? ALL_IND[modalKey].data : [];
  const s = q.toLowerCase().trim();
  modalFiltered = s ? base.filter(r => Object.values(r).some(v => String(v).toLowerCase().includes(s))) : [...base];
  modalPage = 1; renderModal();
}
function sortModal(col) {
  if (modalSortCol === col) modalSortDir *= -1; else { modalSortCol = col; modalSortDir = 1; }
  modalFiltered.sort((a, b) => {
    const av = a[col], bv = b[col], an = parseFloat(av), bn = parseFloat(bv);
    if (!isNaN(an) && !isNaN(bn) && av !== '' && bv !== '') return (an - bn) * modalSortDir;
    return String(av).localeCompare(String(bv), 'es') * modalSortDir;
  });
  document.querySelectorAll('#modal-thead th').forEach(th => { th.classList.remove('sorted'); const a = th.querySelector('.sort-arrow-m'); if (a) a.textContent = '↕'; });
  const cols = ALL_IND[modalKey].cols, ths = document.querySelectorAll('#modal-thead th'), idx = cols.indexOf(col);
  if (ths[idx]) { ths[idx].classList.add('sorted'); const a = ths[idx].querySelector('.sort-arrow-m'); if (a) a.textContent = modalSortDir === 1 ? '↑' : '↓'; }
  modalPage = 1; renderModal();
}
function renderModal() {
  const ind = ALL_IND[modalKey], tbody = document.getElementById('modal-tbody');
  tbody.innerHTML = '';
  const s = (modalPage - 1) * PS, e = s + PS;
  modalFiltered.slice(s, e).forEach(row => {
    const tr = document.createElement('tr');
    ind.cols.forEach(col => {
      const td = document.createElement('td'), v = String(row[col] ?? '');
      if (col === 'semaforo') {
        const cls = v === 'ALERTA ALTA' ? 'b-alert' : v === 'ALERTA MEDIA' ? 'b-warn' : v === 'REVISAR' ? 'b-neutral' : 'b-ok';
        td.innerHTML = `<span class="badge ${cls}">${v || '—'}</span>`;
      } else { td.textContent = v; }
      tr.appendChild(td);
    });
    tbody.appendChild(tr);
  });
  const tot = modalFiltered.length;
  document.getElementById('modal-count').textContent = `Mostrando ${Math.min(s + 1, tot).toLocaleString('es-AR')}–${Math.min(e, tot).toLocaleString('es-AR')} de ${tot.toLocaleString('es-AR')} registros`;
  document.getElementById('modal-sti').style.width = document.getElementById('modal-table').scrollWidth + 'px';
  renderModalPag();
}
function renderModalPag() {
  const tot = modalFiltered.length, tp = Math.ceil(tot / PS);
  const pg = document.getElementById('modal-pg'); pg.innerHTML = '';
  if (tp <= 1) return;
  const add = (lbl, p, active, dis) => {
    const b = document.createElement('button'); b.className = 'pg-btn' + (active ? ' active' : '');
    b.textContent = lbl; b.disabled = dis;
    if (!dis) b.onclick = () => { modalPage = p; renderModal(); };
    pg.appendChild(b);
  };
  add('‹', modalPage - 1, false, modalPage === 1);
  const s = Math.max(1, modalPage - 2), e = Math.min(tp, modalPage + 2);
  if (s > 1) { add('1', 1, false, false); if (s > 2) { const sp = document.createElement('span'); sp.className = 'pg-info'; sp.textContent = '…'; pg.appendChild(sp); } }
  for (let i = s; i <= e; i++) add(i, i, i === modalPage, false);
  if (e < tp) { if (e < tp - 1) { const sp = document.createElement('span'); sp.className = 'pg-info'; sp.textContent = '…'; pg.appendChild(sp); } add(tp, tp, false, false); }
  add('›', modalPage + 1, false, modalPage === tp);
}
function downloadModalExcel() {
  const ind = ALL_IND[modalKey], wb = XLSX.utils.book_new(), ws = XLSX.utils.json_to_sheet(modalFiltered);
  XLSX.utils.book_append_sheet(wb, ws, (ind.num + ' ' + ind.name).substring(0, 31));
  XLSX.writeFile(wb, 'ind_' + ind.num + '.xlsx');
}

// ── INIT ──────────────────────────────────────────────────────────────
buildDrawer();
buildFilters();
buildHead();
applyFilters();
initScrollSync();
"""
APP_JS = r"""/* ============================================================================
   MOTOR + VISTAS MACRO/PLANTA  (pestañas 2 y 3)
   Calcula colores, score y semáforo de grupo EN EL NAVEGADOR a partir de los
   valores crudos + umbrales que vienen en DATOS.meta. Nada precalculado.
   ========================================================================== */
(function(){
const META   = DATOS.meta;
const MESES  = META.meses;
const INDDEF = META.indicadores;
const GRUPOS = META.grupos;
const PLANTAS= DATOS.plantas;
const VALS   = DATOS.valores;
const FAENA  = DATOS.faena;
const GORDER = Object.keys(GRUPOS);
const IND_ORDER = ["01","03","04","06","07","08","10","11","12","09","13","16","05","14","15"];
const RUCAS  = Object.keys(PLANTAS);
const PADRON_TOTAL = 489;   // total padrón SIOCAL (las que no faenaron no están en el JSON)

const MNAMES=['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'];
const mesLbl = m => { const [y,mm]=m.split('-'); return MNAMES[+mm-1]+' '+y.slice(2); };
const MESES_LBL = MESES.map(mesLbl);

const VD='#16a34a', AM='#eab000', RJ='#dc2626', GR='#94a3b8', AZ='#242c4f', NA='#d97706';
const Cc={v:VD,a:AM,r:RJ,g:GR};
const SCOREMAP={r:2,a:0.5,v:0,g:0};

function colorOf(id, v){
  if(v===null||v===undefined) return 'g';
  if(v==='nd')                return 'g';
  const d=INDDEF[id], val=+v;
  if(d.dir==='flag')  return val>=1 ? 'r' : 'v';
  if(d.dir==='bajo'){ if(val<=d.rojo) return 'r'; if(val<=d.amarillo) return 'a'; return 'v'; }
  if(val>=d.rojo) return 'r'; if(val>=d.amarillo) return 'a'; return 'v';
}
function worst(cols){
  const eff=cols.filter(c=>c!=='g');
  if(!eff.length) return 'g';
  if(eff.includes('r')) return 'r';
  if(eff.includes('a')) return 'a';
  return 'v';
}
function faenaMes(ruca,mes){
  const f=FAENA[ruca] && FAENA[ruca][mes]; if(!f) return 0;
  return Object.values(f).reduce((a,b)=>a+(+b||0),0);
}
function faenaBovMes(ruca,mes){ const f=FAENA[ruca]&&FAENA[ruca][mes]; return f?(+f['Bovinos']||0):0; }
function activa(ruca,mes){ return faenaMes(ruca,mes)>0; }

const _cache={};
function serie(ruca){
  if(_cache[ruca]) return _cache[ruca];
  const v=VALS[ruca]||{};
  const col={}, ser={};
  IND_ORDER.forEach(id=>{ col[id]=[]; ser[id]=[]; });
  const grupos={}; GORDER.forEach(g=>grupos[g]=[]);
  const score=[], scoreSem=[], peor=[], activo=[], cabezas=[];
  const fisc = (PLANTAS[ruca].score_claudio==null ? 2 : +PLANTAS[ruca].score_claudio);
  MESES.forEach((m)=>{
    const vm=v[m]||{};
    IND_ORDER.forEach(id=>{
      const raw = (id in vm)? vm[id] : null;
      ser[id].push(raw==='nd'||raw===null||raw===undefined? null : +raw);
      col[id].push(colorOf(id, raw));
    });
    let sc=0; const gcols=[];
    GORDER.forEach(g=>{
      const w=worst(GRUPOS[g].indicadores.map(id=>colorOf(id,(id in vm)?vm[id]:null)));
      grupos[g].push(w); gcols.push(w); sc+=SCOREMAP[w];
    });
    const act=activa(ruca,m);
    if(gcols.every(c=>c==='g') && act) sc+=3;
    scoreSem.push(sc);
    score.push(+(fisc*(sc+1)).toFixed(2));   // Score Total = Fisc × (Semáforo + 1)
    peor.push(worst(gcols)); activo.push(act); cabezas.push(faenaMes(ruca,m));
  });
  return _cache[ruca]={col,ser,grupos,score,scoreSem,peor,activo,cabezas};
}
function faenaEspecies(ruca){
  const out={}; const f=FAENA[ruca]||{};
  MESES.forEach(m=>{ const fm=f[m]||{}; Object.keys(fm).forEach(e=>{ out[e]=out[e]||MESES.map(()=>0); }); });
  Object.keys(out).forEach(e=>{ MESES.forEach((m,i)=>{ out[e][i]=(f[m]&&+f[m][e])||0; }); });
  return out;
}

/* ============================ CHART HELPER ============================= */
const CH={};
function chart(id,cfg){ const el=document.getElementById(id); if(!el) return;
  if(CH[id])CH[id].destroy(); CH[id]=new Chart(el,cfg); }
function hmGrid(id,rows,colorFn,lblw){
  const h=document.getElementById(id); if(!h) return; h.innerHTML='';
  h.style.minWidth='560px'; h.style.gridTemplateColumns=`${lblw}px repeat(${MESES.length},1fr)`;
  h.appendChild(Object.assign(document.createElement('div'),{className:'hmlbl'}));
  MESES_LBL.forEach(m=>{const e=document.createElement('div');e.className='hmmes';e.textContent=m.split(' ')[0];h.appendChild(e);});
  rows.forEach(rw=>{const l=document.createElement('div');l.className='hmlbl';l.textContent=rw.label;h.appendChild(l);
    MESES.forEach((m,i)=>{const e=document.createElement('div');e.className='hmcell';e.style.background=colorFn(rw.key,i);h.appendChild(e);});});
}

/* ============================ FILTROS MACRO ============================= */
// universo de plantas que pasan los filtros macro (por atributos de padrón)
let FILTRO={tam:'',esp:'',act:'',tra:'',reg:'',prov:''};
function pasaFiltro(ruca){
  const p=PLANTAS[ruca];
  if(FILTRO.tam && (p.tamano||'')!==FILTRO.tam) return false;
  if(FILTRO.esp && !(p.especies||[]).includes(FILTRO.esp)) return false;
  if(FILTRO.act && (p.actividad||'')!==FILTRO.act) return false;
  if(FILTRO.tra && (p.transito||'')!==FILTRO.tra) return false;
  if(FILTRO.reg && (p.region||'')!==FILTRO.reg) return false;
  if(FILTRO.prov && (p.provincia||'')!==FILTRO.prov) return false;
  return true;
}
function rucasFiltradas(){ return RUCAS.filter(pasaFiltro); }
// cuántas plantas del padrón total (489) representan el filtro:
// si no hay filtro -> 489; si hay filtro -> solo las filtradas del JSON (no sabemos las inactivas)
function padronRef(){ const hay=Object.values(FILTRO).some(x=>x); return hay? rucasFiltradas().length : PADRON_TOTAL; }

function buildFiltros(){
  const uniq=(f)=>[...new Set(RUCAS.map(r=>PLANTAS[r][f]).filter(x=>x!=null&&x!==''))].sort((a,b)=>String(a).localeCompare(String(b),'es'));
  const esps=[...new Set(RUCAS.flatMap(r=>PLANTAS[r].especies||[]))].sort();
  const fill=(id,vals)=>{ const s=document.getElementById(id); if(!s||s.dataset.filled) return; vals.forEach(v=>s.add(new Option(v,v))); s.dataset.filled='1'; };
  fill('fTam',uniq('tamano')); fill('fEsp',esps); fill('fAct',uniq('actividad'));
  fill('fTra',uniq('transito')); fill('fReg',uniq('region')); fill('fProv',uniq('provincia'));
  const bind=(id,key)=>{ const s=document.getElementById(id); if(s) s.onchange=()=>{FILTRO[key]=s.value; MA=null; renderMacro();}; };
  bind('fTam','tam');bind('fEsp','esp');bind('fAct','act');bind('fTra','tra');bind('fReg','reg');bind('fProv','prov');
  const clr=document.getElementById('fClear'); if(clr) clr.onclick=()=>{ FILTRO={tam:'',esp:'',act:'',tra:'',reg:'',prov:''};
    ['fTam','fEsp','fAct','fTra','fReg','fProv'].forEach(id=>{const s=document.getElementById(id); if(s)s.value='';}); MA=null; renderMacro(); };
}

/* ============================ VISTA MACRO ============================= */
function computeMacro(){
  const POOL=rucasFiltradas();
  const S={}; POOL.forEach(r=>S[r]=serie(r));
  const last=MESES.length-1;
  const actIdx = m => POOL.filter(r=>S[r].activo[m]);
  const actL=actIdx(last);
  const ref=padronRef();
  const scoreProm = m => { const a=actIdx(m); return a.length? a.reduce((s,r)=>s+S[r].score[m],0)/a.length : 0; };
  const pctRojo = m => { const a=actIdx(m); return a.length? 100*a.filter(r=>S[r].peor[m]==='r').length/a.length : 0; };
  const cabezasMes = m => POOL.reduce((s,r)=>s+S[r].cabezas[m],0);
  // KPIs de tipificación (bovino, último mes)
  const bov = POOL.filter(r=>PLANTAS[r].es_bovino);
  const bovTip = bov.filter(r=>(PLANTAS[r].cabezas_tipificadas_mes||0)>0);
  const cabTip = bov.reduce((s,r)=>s+(PLANTAS[r].cabezas_tipificadas_mes||0),0);
  const faenaBov = POOL.reduce((s,r)=>s+faenaBovMes(r,MESES[last]),0);
  const kpi={
    activas:actL.length, padron:ref,
    score:scoreProm(last).toFixed(1), score_d:(scoreProm(last)-scoreProm(0)).toFixed(1),
    rojo:pctRojo(last).toFixed(0), rojo_d:(pctRojo(last)-pctRojo(0)).toFixed(0),
    cabezas:Math.round(cabezasMes(last)).toLocaleString('es-AR'),
    verde:(100*actL.filter(r=>S[r].peor[last]==='v').length/(actL.length||1)).toFixed(0),
    pct_faenando:(100*actL.length/(ref||1)).toFixed(0),
    n_rojo_grupos: actL.reduce((s,r)=>s+GORDER.filter(g=>S[r].grupos[g][last]==='r').length,0),
    pct_faena_tip:(faenaBov>0? 100*cabTip/faenaBov : 0).toFixed(0),
    pct_frig_tip:(bov.length? 100*bovTip.length/bov.length : 0).toFixed(0),
  };
  const estadoFor = key => {
    const out={verde:[],amar:[],rojo:[],gris:[]};
    MESES.forEach((m,i)=>{ const a=actIdx(i); const n=a.length||1;
      const col = r => key==='General'? S[r].peor[i] : S[r].grupos[key][i];
      out.verde.push(100*a.filter(r=>col(r)==='v').length/n);
      out.amar .push(100*a.filter(r=>col(r)==='a').length/n);
      out.rojo .push(100*a.filter(r=>col(r)==='r').length/n);
      out.gris .push(100*a.filter(r=>col(r)==='g').length/n);
    });
    return out;
  };
  const estado={General:estadoFor('General')};
  GORDER.forEach(g=>estado[GRUPOS[g].nombre+' ('+g+')']=estadoFor(g));
  const score_prom=MESES.map((m,i)=>+scoreProm(i).toFixed(2));
  const faena_alerta=MESES.map((m,i)=>{ const tot=cabezasMes(i)||1;
    const exp=POOL.filter(r=>S[r].peor[i]==='r').reduce((s,r)=>s+S[r].cabezas[i],0);
    return +(100*exp/tot).toFixed(1); });
  const heat={};
  IND_ORDER.forEach(id=>{ heat[id]=MESES.map((m,i)=>{
    const ap=actIdx(i).filter(r=>S[r].col[id][i]!=='g'); if(!ap.length) return 0;
    return 100*ap.filter(r=>S[r].col[id][i]==='r').length/ap.length; }); });
  const ranking=IND_ORDER.map(id=>[`${id} · ${INDDEF[id].nombre}`, +heat[id][last].toFixed(0), +(heat[id][last]-heat[id][0]).toFixed(0)])
                          .sort((a,b)=>b[1]-a[1]);
  // Lorenz: AMBAS curvas ordenadas por faena (mayor a menor). Mismo eje X = mismas plantas.
  const ordFaena=actL.slice().sort((a,b)=>S[b].cabezas[last]-S[a].cabezas[last]);
  const totF=ordFaena.reduce((s,r)=>s+S[r].cabezas[last],0)||1;
  const totR=ordFaena.reduce((s,r)=>s+S[r].score[last],0)||1;
  const xs=[0],cf=[0],cr=[0]; let af=0,ar=0;
  const step=Math.max(1,Math.round(ordFaena.length/10));
  for(let k=step;k<=ordFaena.length;k+=step){
    af=ordFaena.slice(0,k).reduce((s,r)=>s+S[r].cabezas[last],0);
    ar=ordFaena.slice(0,k).reduce((s,r)=>s+S[r].score[last],0);
    xs.push(+(100*k/ordFaena.length).toFixed(0)); cf.push(+(100*af/totF).toFixed(1)); cr.push(+(100*ar/totR).toFixed(1));
  }
  if(xs[xs.length-1]!==100){ xs.push(100); cf.push(100); cr.push(100); }
  const lorenz={xs,faena:cf,riesgo:cr};
  const segBy = attr => { const g={}; actL.forEach(r=>{ const k=PLANTAS[r][attr]||'—'; (g[k]=g[k]||[]).push(S[r].score[last]); });
    const o={}; Object.keys(g).sort().forEach(k=>o[k]=+(g[k].reduce((a,b)=>a+b,0)/g[k].length).toFixed(1)); return o; };
  const deltas=POOL.filter(r=>S[r].activo[0]&&S[r].activo[last])
    .map(r=>({r,nombre:PLANTAS[r].razon_social,tr:PLANTAS[r].transito||'—',tam:(PLANTAS[r].tamano||'—').split(' ')[0],
              a:S[r].score[0],b:S[r].score[last],d:+(S[r].score[last]-S[r].score[0]).toFixed(1)}));
  const empeoraron=deltas.filter(x=>x.d>0).sort((a,b)=>b.d-a.d).slice(0,8);
  const mejoraron =deltas.filter(x=>x.d<0).sort((a,b)=>a.d-b.d).slice(0,8);
  return {kpi,estado,score_prom,faena_alerta,heat,ranking,lorenz,
          transito:segBy('transito'),tamano:segBy('tamano'),region:segBy('region'),empeoraron,mejoraron};
}

let MA=null;
function renderMacro(){
  buildFiltros();
  MA=computeMacro();
  const k=MA.kpi;
  const mkpi=(l,val,sub,cls)=>`<div class="kpi"><div class="lbl">${l}</div><div class="val">${val}</div><div class="sub ${cls||'flat'}">${sub||''}</div></div>`;
  const box=document.getElementById('mkpis'); box.className='kpis'; box.style.gridTemplateColumns='repeat(4,1fr)';
  const arrow=v=>v>0?'▲ +'+v:(v<0?'▼ '+v:'– '+v), cls=v=>v>0?'up':(v<0?'down':'flat');
  box.innerHTML=[
    mkpi('Establecimientos activos',`${k.activas} <span style="font-size:12px;color:#64748b">/ ${k.padron}</span>`,'faenaron en el mes'),
    mkpi('Score promedio nacional',k.score,`${arrow(+k.score_d)} vs. inicio`,cls(+k.score_d)),
    mkpi('Plantas con ≥1 grupo rojo',k.rojo+'%',`${arrow(+k.rojo_d)} pp vs. inicio`,cls(+k.rojo_d)),
    mkpi('% del padrón faenando',k.pct_faenando+'%','sobre el total'),
    mkpi('Cabezas faenadas',k.cabezas,'en el mes'),
    mkpi('% faena tipificada',k.pct_faena_tip+'%','bovinos · tipificadas / faenadas'),
    mkpi('% frigoríficos tipificando',k.pct_frig_tip+'%','sobre plantas bovinas'),
    mkpi('Plantas 100% verde',k.verde+'%','sobre activas'),
  ].join('');
  const selE=document.getElementById('selEstado');
  if(!selE.dataset.bound){ selE.onchange=drawEstado; selE.dataset.bound='1'; }
  selE.innerHTML=''; Object.keys(MA.estado).forEach(kk=>selE.add(new Option(kk,kk)));
  drawEstado();
  chart('cScore',{type:'line',data:{labels:MESES_LBL,datasets:[
    {label:'Score promedio',data:MA.score_prom,borderColor:AZ,tension:.3,pointRadius:0,borderWidth:2,yAxisID:'y'},
    {label:'% faena en plantas con rojo',data:MA.faena_alerta,borderColor:RJ,borderDash:[5,4],tension:.3,pointRadius:0,borderWidth:2,yAxisID:'y1'}]},
    options:{plugins:{legend:{position:'bottom',labels:{boxWidth:12}}},scales:{x:{grid:{display:false}},
      y:{position:'left',beginAtZero:true,title:{display:true,text:'Score'},grid:{color:'#eef1f5'}},
      y1:{position:'right',beginAtZero:true,max:100,title:{display:true,text:'% faena'},grid:{display:false}}}}});
  const escM=v=>{if(v<=.5)return'#f1f5f9';if(v<8)return'#fee2e2';if(v<16)return'#fca5a5';if(v<26)return'#f87171';if(v<36)return'#dc2626';if(v<48)return'#b91c1c';return'#7f1d1d';};
  const h=document.getElementById('mheat'); h.innerHTML=''; h.style.minWidth='620px'; h.style.gridTemplateColumns=`180px repeat(${MESES.length},1fr)`;
  h.appendChild(Object.assign(document.createElement('div'),{className:'hmlbl'}));
  MESES_LBL.forEach(m=>{const e=document.createElement('div');e.className='hmmes';e.textContent=m.split(' ')[0];h.appendChild(e);});
  IND_ORDER.forEach(id=>{const l=document.createElement('div');l.className='hmlbl';l.textContent=`${id} · ${INDDEF[id].nombre}`;h.appendChild(l);
    MA.heat[id].forEach(v=>{const e=document.createElement('div');e.className='hmcell';e.style.background=escM(v);e.textContent=Math.round(v);if(v<26)e.style.color=v<=.5?'#94a3b8':'#7f1d1d';h.appendChild(e);});});
  const rk=MA.ranking.slice(0,10);
  chart('cRank',{type:'bar',data:{labels:rk.map(r=>r[0]),datasets:[{data:rk.map(r=>r[1]),backgroundColor:rk.map(r=>r[1]>25?RJ:(r[1]>10?NA:AZ)),borderRadius:3}]},
    options:{indexAxis:'y',plugins:{legend:{display:false},tooltip:{callbacks:{label:c=>` ${c.raw}% en rojo (Δ ${rk[c.dataIndex][2]>0?'+':''}${rk[c.dataIndex][2]} pp)`}}},scales:{x:{grid:{color:'#eef1f5'}},y:{grid:{display:false}}}}});
  chart('cConc',{type:'line',data:{labels:MA.lorenz.xs.map(v=>v+'%'),datasets:[
    {label:'Faena acumulada',data:MA.lorenz.faena,borderColor:AZ,tension:.2,pointRadius:2,borderWidth:2},
    {label:'Riesgo acumulado (mismo orden)',data:MA.lorenz.riesgo,borderColor:RJ,tension:.2,pointRadius:2,borderWidth:2},
    {label:'Reparto proporcional',data:MA.lorenz.xs,borderColor:'#cbd5e1',borderDash:[4,4],pointRadius:0,borderWidth:1}]},
    options:{plugins:{legend:{position:'bottom',labels:{boxWidth:12}}},scales:{x:{title:{display:true,text:'% de establecimientos (ordenados por faena, mayor→menor)'},grid:{display:false}},y:{max:100,title:{display:true,text:'% acumulado'},grid:{color:'#eef1f5'}}}}});
  const seg=(id,obj)=>{const ks=Object.keys(obj);
    chart(id,{type:'bar',data:{labels:ks,datasets:[{data:ks.map(k=>obj[k]),backgroundColor:AZ,borderRadius:3}]},
      options:{plugins:{legend:{display:false}},scales:{x:{grid:{display:false}},y:{beginAtZero:true,grid:{color:'#eef1f5'}}}}});};
  seg('cTra',MA.transito); seg('cTam',MA.tamano); seg('cReg',MA.region);
  const fillT=(id,rows,cls)=>{document.getElementById(id).innerHTML=rows.map(r=>
    `<tr><td><a class="lnk-planta" data-ruca="${r.r}">${r.nombre}</a></td><td><span class="tag">${r.tr[0]}</span></td><td>${r.tam}</td><td class="mono">${r.a}→<b>${r.b}</b></td><td><span class="badge ${cls}">${r.d>0?'+':''}${r.d}</span></td></tr>`).join('');};
  fillT('tEmp',MA.empeoraron,'bup'); fillT('tMej',MA.mejoraron,'bdown');
}
function drawEstado(){ const selE=document.getElementById('selEstado'); const e=MA.estado[selE.value]||MA.estado['General'];
  chart('cEstado',{type:'line',data:{labels:MESES_LBL,datasets:[
    {label:'Verde',data:e.verde,backgroundColor:'rgba(22,163,74,.75)',borderColor:VD,fill:true,tension:.3,pointRadius:0,borderWidth:1},
    {label:'Amarillo',data:e.amar,backgroundColor:'rgba(234,176,0,.75)',borderColor:AM,fill:true,tension:.3,pointRadius:0,borderWidth:1},
    {label:'Rojo',data:e.rojo,backgroundColor:'rgba(220,38,38,.8)',borderColor:RJ,fill:true,tension:.3,pointRadius:0,borderWidth:1},
    {label:'Gris',data:e.gris,backgroundColor:'rgba(148,163,184,.7)',borderColor:GR,fill:true,tension:.3,pointRadius:0,borderWidth:1}]},
    options:{plugins:{legend:{position:'bottom',labels:{boxWidth:12}}},scales:{y:{stacked:true,max:100,grid:{color:'#eef1f5'}},x:{stacked:true,grid:{display:false}}}}});}

/* ============================ VISTA PLANTA ============================= */
let PICK=IND_ORDER.filter(id=>!INDDEF[id].binario).slice(0,4);
let COHSEL=['General'];
const PAL=['#242c4f','#dc2626','#16a34a','#d97706','#0891b2','#7c3aed','#db2777','#65a30d','#ca8a04','#0d9488','#e11d48','#4f46e5','#b45309'];

function plantaOptions(){ return RUCAS.map(r=>({r,p:PLANTAS[r]}))
  .sort((a,b)=>String(a.p.razon_social).localeCompare(String(b.p.razon_social),'es')); }
function fillPlantaSelect(){
  const sel=document.getElementById('selPlanta'); if(sel.dataset.filled) return;
  plantaOptions().forEach(({r,p})=>sel.add(new Option(`${p.razon_social} — SIOCAL ${r} — CUIT ${p.cuit}`, r)));
  sel.dataset.filled='1'; sel.onchange=()=>renderPlanta(sel.value);
  // buscadores separados: filtran las opciones del select
  const qs=document.getElementById('qSiocal'), qr=document.getElementById('qRazon'), qc=document.getElementById('qCuit');
  const applyQ=()=>{ const s=(qs.value||'').toLowerCase().trim(), rz=(qr.value||'').toLowerCase().trim(), c=(qc.value||'').toLowerCase().trim();
    let first=null;
    [...sel.options].forEach(o=>{ const p=PLANTAS[o.value];
      const okS=!s||String(o.value).toLowerCase().includes(s);
      const okR=!rz||String(p.razon_social||'').toLowerCase().includes(rz);
      const okC=!c||String(p.cuit||'').toLowerCase().includes(c);
      const vis=okS&&okR&&okC; o.hidden=!vis; if(vis&&first===null)first=o.value; });
    if(first!==null && (s||rz||c)){ sel.value=first; renderPlanta(first); } };
  [qs,qr,qc].forEach(el=>{ if(el) el.oninput=applyQ; });
}
function renderPlanta(ruca){
  const p=PLANTAS[ruca]; if(!p) return;
  const sp=document.getElementById('selPlanta'); sp.value=ruca; sp.dataset.rendered='1';
  const S=serie(ruca), last=MESES.length-1;
  const esp=faenaEspecies(ruca);
  document.getElementById('ident').innerHTML=`<h1>${p.razon_social}</h1>
    <div class="ruca mono">SIOCAL ${ruca} · CUIT ${p.cuit}</div>
    <div class="chips">
      <span class="chip"><b>Tránsito:</b> ${p.transito||'—'}</span><span class="chip"><b>Tamaño:</b> ${p.tamano||'—'}</span>
      <span class="chip"><b>Región:</b> ${p.region||'—'}</span><span class="chip"><b>Provincia:</b> ${p.provincia||'—'}</span>
      <span class="chip"><b>Especies:</b> ${(p.especies||[]).join(', ')||'—'}</span></div>`;
  const dsc=+(S.score[last]-S.score[last-1]).toFixed(1);
  const fl=S.cabezas[last], fp=S.cabezas[last-1]||0, fd=fp?Math.round((fl-fp)/fp*100):0;
  // faena: sube -> verde, baja -> rojo
  // OJO: en el CSS .up=rojo y .down=verde. Para faena queremos sube->verde, baja->rojo:
  const faenaCls = fd>0?'down':(fd<0?'up':'flat');
  const faenaTxt = fp? ((fd>0?'▲ +':(fd<0?'▼ ':'– '))+Math.abs(fd)+'% vs. mes ant.') : 's/comp.';
  const fisc = (p.score_claudio==null?2:+p.score_claudio);
  const fiscTxt = p.fisc_imputado? `${fisc} <span style="font-size:10px;color:#94a3b8">(imputado)</span>` : fisc;
  // tipificador habilitado
  let tipTxt, tipColor='#64748b';
  if(!p.es_bovino){ tipTxt='No corresponde'; }
  else if((p.cabezas_tipificadas_mes||0)===0){ tipTxt='No tipifica'; }
  else { // buscar último mes con dato del indicador 11
    let flag=null; for(let i=last;i>=0;i--){ const c=S.col['11'][i]; if(c!=='g'){ flag=c; break; } }
    if(flag==='r'){ tipTxt='No habilitado'; tipColor=RJ; } else if(flag==='v'){ tipTxt='Habilitado'; tipColor=VD; } else { tipTxt='Sin dato'; }
  }
  const ultInsp = p.ultima_inspeccion || 's/d';
  document.getElementById('ukpis').innerHTML=[
    `<div class="kpi"><div class="lbl">Score Total actual</div><div class="val">${S.score[last]}</div><div class="sub ${dsc>0?'up':(dsc<0?'down':'flat')}">${dsc>0?'▲ +'+dsc:(dsc<0?'▼ '+dsc:'– 0')} vs. mes ant.</div></div>`,
    `<div class="kpi"><div class="lbl">Scoring Fiscalización</div><div class="val">${fiscTxt}</div><div class="sub flat">mochila (1–4)</div></div>`,
    `<div class="kpi"><div class="lbl">Faena último mes</div><div class="val">${Math.round(fl).toLocaleString('es-AR')}</div><div class="sub ${faenaCls}">${faenaTxt}</div></div>`,
    `<div class="kpi"><div class="lbl">Tipificador habilitado</div><div class="val" style="font-size:17px;font-family:Lora;color:${tipColor}">${tipTxt}</div></div>`,
    `<div class="kpi"><div class="lbl">Última inspección</div><div class="val" style="font-size:17px;font-family:Lora">${ultInsp}</div></div>`,
  ].join('');
  chart('uScore',{type:'line',data:{labels:MESES_LBL,datasets:[{data:S.score,borderColor:AZ,backgroundColor:'rgba(36,44,79,.08)',fill:true,tension:.3,pointRadius:3,borderWidth:2}]},options:{plugins:{legend:{display:false}},scales:{y:{beginAtZero:true,grid:{color:'#eef1f5'}},x:{grid:{display:false}}}}});
  const espKeys=Object.keys(esp);
  chart('uFaena',{type:'bar',data:{labels:MESES_LBL,datasets:espKeys.map((e,i)=>({label:e,data:esp[e],backgroundColor:PAL[i%PAL.length]}))},options:{plugins:{legend:{position:'bottom',labels:{boxWidth:12}}},scales:{x:{stacked:true,grid:{display:false}},y:{stacked:true,grid:{color:'#eef1f5'}}}}});
  hmGrid('uSem',GORDER.map(g=>({key:g,label:`${g} · ${GRUPOS[g].nombre}`})),(g,i)=>Cc[S.grupos[g][i]],180);
  hmGrid('uHeat',IND_ORDER.map(id=>({key:id,label:`${id} · ${INDDEF[id].nombre}`})),(id,i)=>Cc[S.col[id][i]],180);
  drawPick(ruca); buildCoh(ruca); drawComp(ruca); drawDesg(ruca);
}
function drawPick(ruca){ const S=serie(ruca);
  chart('uInd',{type:'line',data:{labels:MESES_LBL,datasets:PICK.map((id,i)=>({label:`${id} · ${INDDEF[id].nombre}`,data:S.ser[id],borderColor:PAL[i%PAL.length],tension:.3,pointRadius:0,borderWidth:2,spanGaps:true}))},
    options:{plugins:{legend:{position:'bottom',labels:{boxWidth:10,font:{size:9}}}},scales:{x:{grid:{display:false}},y:{beginAtZero:true,grid:{color:'#eef1f5'}}}}});}
function buildPick(){ const pick=document.getElementById('uPick'); if(pick.dataset.filled) return; pick.dataset.filled='1';
  IND_ORDER.filter(id=>!INDDEF[id].binario).forEach(id=>{ const lb=document.createElement('label');
    lb.innerHTML=`<input type="checkbox" ${PICK.includes(id)?'checked':''}>${id} · ${INDDEF[id].nombre}`;
    if(PICK.includes(id))lb.classList.add('on');
    lb.querySelector('input').onchange=e=>{ if(e.target.checked){PICK.push(id);lb.classList.add('on');}else{PICK=PICK.filter(x=>x!==id);lb.classList.remove('on');}
      drawPick(document.getElementById('selPlanta').value); };
    pick.appendChild(lb); }); }
function buildCoh(ruca){ const p=PLANTAS[ruca], coh=document.getElementById('uCoh'); coh.innerHTML='';
  const opts=[['General','General'],['Mismo tránsito','tr:'+p.transito],['Mismo tamaño','tam:'+p.tamano],['Misma región','reg:'+p.region]];
  opts.forEach(([lbl,key])=>{ const on=COHSEL.includes(key); const lb=document.createElement('label');
    lb.innerHTML=`<input type="checkbox" ${on?'checked':''}>${lbl}`; if(on)lb.classList.add('on');
    lb.querySelector('input').onchange=e=>{ if(e.target.checked){ if(key==='General')COHSEL=['General']; else {COHSEL=COHSEL.filter(x=>x!=='General'); COHSEL.push(key);} }
      else COHSEL=COHSEL.filter(x=>x!==key);
      if(!COHSEL.length)COHSEL=['General']; buildCoh(ruca); drawComp(ruca); };
    coh.appendChild(lb); }); }
function cohortMean(key){ const last=MESES.length-1;
  const match=r=>{ if(key==='General')return true; const [t,v]=key.split(':'); const p=PLANTAS[r];
    return t==='tr'?p.transito===v : t==='tam'?p.tamano===v : p.region===v; };
  return MESES.map((m,i)=>{ const rs=RUCAS.filter(r=>serie(r).activo[i]&&match(r)); if(!rs.length)return null;
    return +(rs.reduce((s,r)=>s+serie(r).score[i],0)/rs.length).toFixed(2); }); }
function drawComp(ruca){ const S=serie(ruca);
  const ds=[{label:'Esta planta',data:S.score,borderColor:AZ,borderWidth:3,tension:.3,pointRadius:3}];
  const names={General:'General','tr':'Mismo tránsito','tam':'Mismo tamaño','reg':'Misma región'};
  COHSEL.forEach((key,i)=>{ const lbl=key==='General'?'Promedio general':names[key.split(':')[0]];
    ds.push({label:lbl,data:cohortMean(key),borderColor:PAL[(i+1)%PAL.length],borderDash:[5,4],borderWidth:2,tension:.3,pointRadius:0}); });
  chart('uComp',{type:'line',data:{labels:MESES_LBL,datasets:ds},options:{plugins:{legend:{position:'bottom',labels:{boxWidth:12}}},scales:{x:{grid:{display:false}},y:{beginAtZero:true,grid:{color:'#eef1f5'}}}}});}
function drawDesg(ruca){ const S=serie(ruca), last=MESES.length-1, cont=document.getElementById('uDg'); cont.innerHTML='';
  const fmt=(id)=>{ const raw=(VALS[ruca]&&VALS[ruca][MESES[last]]&&(id in VALS[ruca][MESES[last]]))?VALS[ruca][MESES[last]][id]:null;
    if(raw===null||raw===undefined)return 'n/a'; if(raw==='nd')return 'nd';
    if(INDDEF[id].dir==='flag')return (+raw>=1?'Sí':'No'); if(INDDEF[id].dir==='conteo')return String(+raw);
    return (+raw).toFixed(1)+'%'; };
  GORDER.forEach(g=>{ const d=document.createElement('div'); d.className='dgcol';
    let html=`<h3>${g} · ${GRUPOS[g].nombre}</h3>`;
    GRUPOS[g].indicadores.forEach(id=>{ const c=S.col[id][last];
      html+=`<div class="dgi"><span class="nm">${id} · ${INDDEF[id].nombre}</span><span class="pill ${c}">${fmt(id)}</span></div>`; });
    d.innerHTML=html; cont.appendChild(d); }); }

/* ============================ TABS + WIRING ============================= */
function showTab(name){
  document.querySelectorAll('.topnav .tab').forEach(t=>t.classList.toggle('on',t.dataset.v===name));
  document.querySelectorAll('.pane').forEach(v=>v.classList.toggle('on',v.id==='pane-'+name));
  document.getElementById('ctrl-toolbar').style.display = name==='control'?'flex':'none';
  if(name==='macro') renderMacro();
  if(name==='planta'){ fillPlantaSelect(); buildPick(); const sp=document.getElementById('selPlanta'); if(!sp.dataset.rendered){ renderPlanta(sp.value||sp.options[0].value); } }
}
function goPlanta(ruca){ ruca=String(ruca);
  if(!PLANTAS[ruca]){ alert('Este establecimiento está en el padrón de control pero no tiene datos mensuales de indicadores para la vista por planta.'); return; }
  fillPlantaSelect(); buildPick(); showTab('planta'); renderPlanta(ruca); window.scrollTo(0,0); }
document.querySelectorAll('.topnav .tab').forEach(t=>t.onclick=()=>showTab(t.dataset.v));
document.addEventListener('click',e=>{ const a=e.target.closest('.lnk-planta'); if(a&&a.dataset.ruca){ e.preventDefault(); goPlanta(a.dataset.ruca); } });
window.goPlanta=goPlanta;
})();
"""
IND_META_JSON = r"""{
 "Ind01_DTE_Cerr_Amp": {
  "num": "01",
  "name": "DTE Cerrados/Amparados",
  "cols": [
   "establecimiento_ruca",
   "cantidad_amparada_total",
   "cantidad_cierre_total",
   "01_pct_cerrado_amparado",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind03_Faltante": {
  "num": "03",
  "name": "DTE con Faltante",
  "cols": [
   "establecimiento_ruca",
   "total_dte",
   "dte_con_faltante",
   "cabezas_faltante_total",
   "03_pct_dte_con_faltante",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind05_Corral_vs_Ingresos": {
  "num": "05",
  "name": "Exceso Cap. Corral",
  "cols": [
   "establecimiento_ruca",
   "razon_social",
   "cuit_titular",
   "fecha",
   "capacidad_corral",
   "cabezas_ingresadas",
   "alarma"
  ]
 },
 "Ind06_Emergencia": {
  "num": "06",
  "name": "Emergencias",
  "cols": [
   "establecimiento_ruca",
   "total",
   "emergencias",
   "06_pct_emergencia_mes_actual",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind07_Posdatadas": {
  "num": "07",
  "name": "Autorizaciones Posdatadas",
  "cols": [
   "establecimiento_ruca",
   "total",
   "posdatadas",
   "07_pct_posdatadas",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind08_AproTardia": {
  "num": "08",
  "name": "Aprobaciones Tardías",
  "cols": [
   "establecimiento_ruca",
   "total_autorizaciones",
   "autorizaciones_tardias",
   "08_pct_aprobacion_tardia",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind09_Demora": {
  "num": "09",
  "name": "DTEs con Demora",
  "cols": [
   "establecimiento_ruca",
   "casos_demora_mayor_3d",
   "promedio_dias_demora",
   "total_dte_cruzados",
   "09_pct_dte_con_demora",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind10_Denticion_Unif": {
  "num": "10",
  "name": "Dentición Uniforme",
  "cols": [
   "establecimiento_ruca",
   "total_tropas_evaluadas",
   "tropas_homogeneas",
   "10_pct_tropas_denticion_uniforme",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind04_Coincidencia_Sexo": {
  "num": "04",
  "name": "Sexo DTE",
  "cols": [
   "establecimiento_ruca",
   "total_cabezas_cruzadas",
   "cabezas_no_coinciden",
   "04_pct_no_coinciden_sexo",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind11_Tipif_Habilitado": {
  "num": "11",
  "name": "Tipificador Habilitado",
  "cols": [
   "establecimiento_ruca",
   "total_registros",
   "registros_no_habilitados",
   "11_tipificador_no_habilitado",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind12_Entropia_Tipif": {
  "num": "12",
  "name": "Entropía Tipificadores",
  "cols": [
   "establecimiento_ruca",
   "razon_social",
   "numero_tipificador",
   "total_animales",
   "A0",
   "A1",
   "A2",
   "A3",
   "A4",
   "B0",
   "B1",
   "B2",
   "B3",
   "B4",
   "C0",
   "C1",
   "C2",
   "C3",
   "C4",
   "D0",
   "D1",
   "D2",
   "D3",
   "D4",
   "E0",
   "E1",
   "E2",
   "E3",
   "E4",
   "entropia",
   "confianza_muestral",
   "score_alerta",
   "semaforo"
  ]
 },
 "Ind13_Faena_vs_Ingreso": {
  "num": "13",
  "name": "Faena vs Ingreso",
  "cols": [
   "establecimiento_ruca",
   "total_dte",
   "cabezas_ingresadas_total",
   "cabezas_faenadas_total",
   "13_pct_faenado_sobre_ingresado",
   "cuit_titular",
   "razon_social"
  ]
 },
 "Ind15_Stock_vs_Camara": {
  "num": "15",
  "name": "Sobrestock Cámara",
  "cols": [
   "establecimiento_ruca",
   "razon_social",
   "fecha",
   "capacidad_camara_total",
   "stock_medias_reses",
   "alarma"
  ]
 },
 "Ind16_Faena_vs_Existencias": {
  "num": "16",
  "name": "Faena sin Existencias",
  "cols": [
   "establecimiento_ruca",
   "dias_con_faena",
   "dias_con_existencias",
   "dias_sin_declarar",
   "16_pct_faena_sin_existencias",
   "razon_social",
   "cuit_titular"
  ]
 },
 "Ind14_Romaneo_vs_Cupo": {
  "num": "14",
  "name": "Romaneo vs Cupo",
  "cols": [
   "establecimiento_ruca",
   "cuit_titular",
   "razon_social",
   "especie",
   "tipo_cupo",
   "cupo",
   "faena_calculada",
   "diferencia",
   "excedido"
  ]
 }
}"""


## Carga de datos + armado del HTML

In [ ]:
# ===================== 1) HELPERS =====================
def _clean_nan(o):
    if isinstance(o, float) and math.isnan(o): return None
    if isinstance(o, dict): return {k:_clean_nan(v) for k,v in o.items()}
    if isinstance(o, list): return [_clean_nan(x) for x in o]
    return o

def _records(df):
    df = df.where(pd.notnull(df), None)
    out=[]
    for r in df.astype(object).to_dict("records"):
        out.append({k:(None if (isinstance(v,float) and math.isnan(v)) else v) for k,v in r.items()})
    return out

def _js(name,obj): return "const %s = %s;" % (name, json.dumps(obj, ensure_ascii=False))
def _sheet(n): return pd.read_excel(XLSX, sheet_name=n, engine="calamine", dtype=str)

# ===================== 2) JSON (Macro / Planta) =====================
with open(JSON_, encoding="utf-8") as f:
    DATOS = _clean_nan(json.load(f))

# ===================== 3) EXCEL (Control) =====================
GROUP_COLS = ["G1 · Ingreso / Trazabilidad","G2 · Autorización de Faena","G3 · Tipificación","G4 · Balance / Volumen","G5 · Capacidad Física"]
GRISES = {"G1":"grises · G1","G2":"grises · G2","G3":"grises · G3","G4":"grises · G4","G5":"grises · G5"}
# Orden de columnas: los 3 scores juntos y al final, con Total Grises antes.
SHOW_COLS = ["Establecimiento SIOCAL","Razón Social","CUIT","Provincia","Actividad","Especies","Estado","Total Cabezas Ingresadas","Última Inspección","Tránsito","Tamaño","Región"]+GROUP_COLS+["Total Grises","Score Fiscalización","Score Semáforo","Score Total"]
SHEET_ORDER = ["Ind01_DTE_Cerr_Amp","Ind03_Faltante","Ind05_Corral_vs_Ingresos","Ind06_Emergencia","Ind07_Posdatadas","Ind08_AproTardia","Ind09_Demora","Ind10_Denticion_Unif","Ind11_Tipif_Habilitado","Ind12_Entropia_Tipif","Ind13_Faena_vs_Ingreso","Ind16_Faena_vs_Existencias","Ind04_Coincidencia_Sexo","Ind15_Stock_vs_Camara","Ind14_Romaneo_vs_Cupo"]

# python-calamine se instala desde requirements.txt.
sem = _sheet("Semáforo por Grupo")
des = _sheet("Desglose por Grupo")

# ALL_DATA = Semáforo por Grupo (+ Total Grises = suma de grises por grupo)
ALL_DATA=[]
for r in _records(sem):
    tg=0
    for gc in GRISES.values():
        try: tg += int(float(r.get(gc) or 0))
        except: pass
    row={c:(r.get(c) if r.get(c) is not None else "") for c in SHOW_COLS if c!="Total Grises"}
    row["Total Grises"]=str(tg)
    ALL_DATA.append(row)

IND_DES_COLS=[c for c in des.columns if c[:2].isdigit()]
DES_DATA={ str(r.get("Establecimiento SIOCAL")): {ic:(r.get(ic) or "GRIS") for ic in IND_DES_COLS} for r in _records(des) }

IND_META=json.loads(IND_META_JSON)
ALL_IND={}
for key in SHEET_ORDER:
    m=IND_META[key]; df=_sheet(key)
    ALL_IND[key]={"num":m["num"],"name":m["name"],"cols":m["cols"],"data":_records(df)}

FILT_COLS=["Provincia","Actividad","Estado","Tránsito","Tamaño","Región"]
FILTROS_OPT={c:sorted({r[c] for r in ALL_DATA if r.get(c)}) for c in FILT_COLS}
_esp=set()
for r in ALL_DATA:
    for s in str(r.get("Especies") or "").split(","):
        s=s.strip()
        if s: _esp.add(s)
ESPECIES_ALL=sorted(_esp)
_years=set()
for r in ALL_DATA:
    ui=r.get("Última Inspección") or ""
    if len(ui)>=4 and ui[:4].isdigit(): _years.add(int(ui[:4]))
    elif len(ui)>=10 and ui[6:10].isdigit(): _years.add(int(ui[6:10]))
YEARS=sorted(_years)
MONTHS={str(i+1):n for i,n in enumerate(['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'])}

# inyectar PADRON_TOTAL en el JS del motor
APP_JS_FINAL = APP_JS.replace("const PADRON_TOTAL = 489;", "const PADRON_TOTAL = %d;" % PADRON_TOTAL)

# ===================== 4) ARMADO DEL HTML =====================
BODY_CONTROL = re.sub(r"<header>.*?</header>", "", BODY_TAB1.replace("__LOGO__", LOGO), flags=re.S)

HTML = ("""<!DOCTYPE html><html lang="es"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Tablero de Fiscalización de Carnes</title>
<link href="https://fonts.googleapis.com/css2?family=Montserrat:wght@300;400;500;600&family=Lora:wght@600&family=Roboto+Mono:wght@400;500&display=swap" rel="stylesheet">
<script src="https://cdn.sheetjs.com/xlsx-0.20.1/package/dist/xlsx.full.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<style>
""" + CSS_SHELL + """
""" + CSS_TAB1 + """
""" + CSS_MM + """
</style></head><body>

<div class="apphdr">
  <img src=\"""" + LOGO + """\" alt="SAGYP">
  <div><div class="t">Tablero de Fiscalización de Carnes</div>
  <div class="s">Subsecretaría de Mercados Agroalimentarios e Inserción Internacional · SAGYP</div></div>
</div>
<div class="topnav">
  <button class="tab on" data-v="control">Tablero de Control</button>
  <button class="tab" data-v="macro">Vista Macro · Nacional</button>
  <button class="tab" data-v="planta">Vista por Planta</button>
</div>
<div id="ctrl-toolbar">
  <button class="hamburger" id="hamburger" onclick="toggleDrawer()"><span></span><span></span><span></span></button>
  <button class="btn-dl" onclick="downloadExcel()">⬇ Descargar Excel</button>
</div>

<div class="pane on" id="pane-control">
""" + BODY_CONTROL + """
</div>
<div class="pane" id="pane-macro"><div class="mm">
""" + MACRO_MK + """
</div></div>
<div class="pane" id="pane-planta"><div class="mm">
""" + PLANTA_MK + """
</div></div>

<script>
""" + "\n".join([_js("ALL_DATA",ALL_DATA), _js("ALL_IND",ALL_IND), _js("DES_DATA",DES_DATA),
                 _js("FILTROS_OPT",FILTROS_OPT), _js("SHOW_COLS",SHOW_COLS), _js("SHEET_ORDER",SHEET_ORDER),
                 _js("ESPECIES_ALL",ESPECIES_ALL), _js("YEARS",YEARS), _js("MONTHS",MONTHS), _js("DATOS",DATOS)]) + """
</script>
<script>
""" + CODE_TAB1 + """
</script>
<script>
""" + APP_JS_FINAL + """
</script>
</body></html>""")

with open(OUT, "w", encoding="utf-8") as f:
    f.write(HTML)

print("OK ->", OUT, "|", round(len(HTML)/1024/1024,2), "MB")
print("Control:", len(ALL_DATA), "| Indicadores:", len(ALL_IND), "| Plantas:", len(DATOS["plantas"]))

print("Abrí este archivo en tu navegador:", os.path.abspath(OUT))
